# `C04` — Cancellation

Scenario **C04** of the Track A certification ladder (card `OMIE-CERT-C04-1`, [`agile-plan.md`](../../docs/omiecid/agile-plan.md); canonical spec [`scenario-ladder.md`](../../docs/omiecid/scenario-ladder.md) §C4). Oracle kind: **exact-parametric**.

**Terms used below** (defined once in `scenario-ladder.md`, "What eager, jitted and assertion mean here"): *eager* runs every JAX operation immediately (`jax.disable_jit()`), so a human can inspect each step; *jitted* traces and compiles the whole run (`jax.jit`), the mode training uses; the two must agree. An *assertion* records a failure and the notebook carries on: every check is evaluated, and the verdict comes at the end. Every pass/fail assertion here is an **exact integer equality** on order ids, prices in ticks (0.01 EUR/MWh) and quantities in ticks (0.1 MW); euro figures are shown, never asserted.

## 1. Purpose

**Purpose.** Removing a resting order must remove exactly that order, produce no trade, and — the
non-obvious part — leave **the queue order of the survivors unchanged**. That last clause is the
reason this scenario is not trivial: the removal compacts a fixed-size array, and a compaction that
swaps the vacated slot with the last element (a normal, efficient implementation choice) silently
reorders the survivors. Every quantity would still conserve, no trade would be wrong, and C3's
priority guarantee would quietly stop holding for every order behind a cancellation. It is asserted
by identity, not by count.

The second claim concerns icebergs, and it reaches into a different subsystem. Cancelling an iceberg
must release its side-channel slot and make its hidden remainder genuinely disappear — because a
later fill-or-kill pre-check reads that side-channel directly, and a stale slot would report phantom
hidden volume. That is not a hypothetical failure mode: it is the shape OQ-20 and OQ-23 already
document. So the scenario cancels an iceberg and then submits a fill-or-kill sized to succeed *only
if* the cancelled hidden volume were still visible to the pre-check, and requires it to be rejected.

**Why modification is excluded, recorded so the gap is visible.** §5.3 makes modification reset an
order's timestamp — *"a hard rule from the XBID system"* — and `modify_order` is implemented (as
cancel-then-resubmit). Excluding it is a **scope decision, not an absence**, and it leaves one hard
rule uncertified. The certificate must say so in those terms rather than letting §5.3 read as
covered.

*(Verbatim from `scenario-ladder.md` §C4.)*

**Certifies** (card): §5.1's branch `CANCELLED (by agent; GTD expiry; GCT; market halt)`, quoted whole; §6.1 priority preservation across a removal; iceberg side-channel release.

**How the claim is pinned.** Two constructions. In both, the cancellation is an engine call (`cancel_order`) made immediately **before the last tape row** is submitted.

*`C04.cancellation` (the ladder).* $k \in [2,5]$ *touch* orders rest at one price, all of size $q$, entered one after another. One to three *decoys* rest at worse prices, beyond the aggressor's limit. One touch order, the **victim**, is cancelled. Then an aggressor arrives on the other side, sized $j \cdot q$ with $1 \le j \le k-1$. Per run the notebook asserts, by order id:

1. **the cancellation removes exactly the victim**: the book just before the aggressor equals the book just after the last resting row, minus the victim's row. Every other row keeps its price, quantity, order id, owner and stored timestamp. The victim's fate is one `cancellation` of its full size, and it never fills;
2. **the cancellation prints no trade**: the trade tape is byte-identical across the cancel;
3. **the survivors keep their queue**: their stored timestamps are still unique and strictly increasing in entry order;
4. **the filled set** is exactly the $j$ earliest survivors, and the **counterparty sequence** (passive id, aggressor id, price, quantity) follows their entry order. A queue that takes the right orders in the wrong order also fails;
5. **the book afterwards**: the other $k-1-j$ survivors and every decoy rest untouched at full size; the victim and the aggressor are absent.

*`C04.iceberg_cancel_fok` (the iceberg).* A standard iceberg (PPD = 0) rests with total $T$ and peak $P$, followed by $n_{vis} \in [0,3]$ visible NON orders at its price. The iceberg is cancelled. Then a FOK arrives on the other side, priced through, and asks $R$ with $V_{vis} < R \le V_{vis} + T$: more than the visible orders hold, and no more than they hold together with the iceberg. Asserted: before the cancel the iceberg shows $P$ and hides $T - P$. After the cancel it shows and hides nothing, and its side-channel slot is released. The cancel prints no trade. The FOK **fills nothing**, does not rest, and leaves the book unchanged. A **no-cancel control** of every draw runs the same tape without the cancel, and the FOK must then fill exactly $R$. The control is what makes the rejection mean "the cancelled volume is gone" rather than "the FOK never fills".

**Two derived ladder runs** make the compaction claim falsifiable (finding F1). `decoys_first` re-enters the decoys before the touch orders, on every draw. The **victim sweep** cancels each touch order in turn, on the drawn tape and on `decoys_first`, for the first `SWEEP_DRAWS` draws.

**Does not certify.**
- §5.3 order modification and its timestamp reset — **excluded by scope decision, though implemented** (`modify_order`, as cancel-then-resubmit). One XBID hard rule goes uncertified: *"Modifying price, quantity, or execution restriction of an order resets its timestamp (loses time priority)."* (§5.3, quoted in section 2).
- The other causes on the same §5.1 branch: GTD expiry and GCT (C11), and market halt (not modelled). Only cancellation *by agent* is exercised.
- Cancelling a partially executed order, a decoy, or an iceberg after some of its slices have traded; two cancellations in one window.
- How the engine reports a killed FOK (`accepted` or rejected): the spec's status vocabulary is not the engine's (finding F3). The FOK pre-check projection itself, ICEwPI included, is C10's.
- Which array slot a cancelled order leaves behind: storage is not a market rule. Only the observable queue (who fills, in what order) is asserted.

**Findings recorded at K2** (from the specification inputs only; no engine source was read).
- **F1 — the drawn ladder alone cannot see a swap-with-last compaction.** The generator enters the decoys **after** the touch orders. Suppose the book stored rows in entry order and filled same-price rows in array order. A swap-with-last removal would then move the **last decoy**, not a touch order, into the victim's slot. The touch queue would keep its order and every drawn run would pass. Hence the `decoys_first` run on every draw (the last resting row is then the last touch order) and the victim sweep. Section 7 asserts that on every sweep draw with $k \ge 3$, the swap-with-last model disagrees with the rule on at least one `decoys_first` sweep run. It also reports how often the model disagrees on the drawn tapes (expected: never). The model is a hypothesis about storage, used only to size the dataset; it is not a claim about the engine.
- **F2 — not every iceberg draw is a strong test.** With $n_{vis} = 0$ the resting side is empty after the cancel. A pre-check that still counted the cancelled volume would find nothing to fill against, so unless the stale slot also manufactured fills, the FOK prints nothing either way. With $R \le V_{vis} + P$, the iceberg's *visible* peak alone could have satisfied the FOK, so the draw tests the removal of the visible slice more than that of the hidden remainder. The card's claim ("a FOK that only its hidden volume could have satisfied") is met by the class $n_{vis} \ge 1$ and $R > V_{vis} + P$. Section 7 asserts that this class is covered and reports the per-class counts.
- **F3 — the killed FOK's status.** §5.1 places an unmatched FOK under `ACCEPTED → CANCELLED (IOC/FOK unmatched; …)`, while the card says the pre-check "rejects". The asserted observable is the same either way: zero fills, no book entry, no book change. The record's fate passes if it is *either* a validator rejection *or* a `fok_kill` of the full $R$. Which of the two occurred is printed, not asserted.
- **F4 — slot release is an engineering claim.** The market rule is that the cancelled iceberg's remaining quantity is gone (§5.1; §4.2.1 Rule 1 counts *remaining* quantity), and this is asserted on the engine's visible and hidden quantities. "Release its side-channel slot" (card; `scenario-ladder.md` §C4; the class of OQ-20) is asserted on the record's live side-channel count `n_active`: one live iceberg before the cancel, none after. It is labelled an engineering invariant, not a market-spec rule.
- **F5 — the tape's timestamp column** (C03's F2). Every derived tape keeps the timestamp column by slot, so it still grows down the tape, and entry order is the same under either reading of where the engine's stamps come from.
- **F6 — source version.** The card names `market-spec.md` v2.12. The source is now v2.13, whose change (OQ-31, §4.2.1 Rule 3) touches no passage quoted here. Every quote is checked against v2.13.

## 2. The rule, quoted

The governing passages, quoted **verbatim** and checked at run time against the document each one names (`notebooks.certification.cert_support.check_quotes`): [`market-spec.md`](../../docs/omiecid/market-spec.md) unless `open-questions.md` or `scenario-ladder-review.md` is named. A quote that has drifted is flagged **DRIFT** below and recorded as `quote_drift`: the notebook still runs to the end and ends `NOT CERTIFIED`. The regulatory sources are shown as citations, not machine-checked.

How the quotes ground the expectation of section 5:

- **What a cancellation is (§5.1, §9.1).** An order `IN_BOOK` can leave the book as `CANCELLED (by agent; …)`. Cancelling takes only an order id and is possible only while the order is active in the book. Every victim here is an untouched resting order, so the cancel is valid. Its effect is the order's removal: nothing else in the state model moves.
- **No trade from the cancel (§6.1, §6.2 items 1–2; Matching Algorithm §4.1).** A trade needs a buy and a sell that cross. When the cancel happens, every resting row is on one side, so no pair can cross, before or after a removal. The Matching Algorithm re-checks for matches after every book update, deletions included, and finds none here.
- **The survivors keep their queue (§5.2, §6.1).** A timestamp is given on entry, and a **new** one only on modification, reactivation or a new iceberg slice. Another order's cancellation is none of these, so every survivor keeps its stamp, and with it its rank: best price first, then earliest timestamp.
- **Who the aggressor takes (§6.1, §6.2 items 1, 3; §4.5).** The aggressor walks the remaining resting side in rank order, each fill at the maker's limit, until it is used up. It is sized $j \cdot q$, so it takes exactly $j$ whole survivors and does not rest. The decoys lie beyond its limit.
- **The cancelled iceberg (§4.2, §4.5, §4.2.1 Rule 1).** Before the cancel only the peak is shown. The FOK pre-check counts an iceberg's *remaining* quantity, and a cancelled iceberg has none. What remains is the visible orders' $V_{vis} < R$, so the FOK cannot be filled in full. §4.5 says a FOK is never partly filled and never enters the book: it executes nothing.
- **The control (§4.2.1 Rules 1–2).** Without the cancel the pre-check sees $V_{vis} + T \ge R$ at a crossing price, and one FOK can take several slices in one match cycle, so it fills exactly $R$.
- **Why a stale slot matters (OQ-20; review §2).** The FOK pre-check reads the iceberg side-channel directly, so a stale slot would over-report hidden volume.

In [ ]:
import time

T0 = time.time()   # the measured wall clock (results: wall_clock_s) starts here

from IPython.display import Markdown, display

from notebooks.certification.cert_support import (
    Certification, Citation, SpecQuote, check_quotes, render_quotes,
)

QUOTES = (
    # --- §5.1: the cancellation branch C04 certifies, quoted whole -------------------------------
    SpecQuote(
        "market-spec.md", "§5.1",
        "└─ IN_BOOK […] ├─ CANCELLED (by agent; GTD expiry; GCT; market halt)",
        note="The branch certified: an order IN_BOOK leaves the book as CANCELLED. Only 'by agent' is exercised "
             "here; GTD expiry and GCT are C11's, market halt is not modelled.",
    ),
    SpecQuote(
        "market-spec.md", "§5.1",
        "├─ CANCELLED (IOC/FOK unmatched; GTD expired; GCT reached; halt/hibernation)",
        note="The killed FOK of the iceberg draw: an unmatched FOK ends CANCELLED (finding F3 on 'rejected').",
    ),
    SpecQuote(
        "market-spec.md", "§9.1",
        "| Cancel order | order_id | Only possible if order is active in book |",
        note="The action: an order id, valid while the order rests. Every victim is an untouched resting order.",
    ),
    # --- §6.1: the ranking the survivors must keep ------------------------------------------------
    SpecQuote(
        "market-spec.md", "§6.1",
        "**Buy side:** ranked highest price first; ties broken by earliest timestamp.",
        note="Ranking, buy side. Exercised by every draw whose resting side is BUY.",
    ),
    SpecQuote(
        "market-spec.md", "§6.1",
        "**Sell side:** ranked lowest price first; ties broken by earliest timestamp.",
        note="Ranking, sell side. Exercised by every draw whose resting side is SELL.",
    ),
    SpecQuote(
        "market-spec.md", "§6.1",
        "A match occurs when best bid price ≥ best ask price.",
        note="At the cancel every resting row is on one side: no bid meets an ask, so the cancel cannot match.",
    ),
    SpecQuote(
        "market-spec.md", "§6.1",
        "This is a hard rule mandated by ACER Decision 11/2024 Art. 5 and CACM Regulation Art. 51.",
        note="Provenance of the ranking rule the survivors must keep.",
    ),
    # --- §5.2: why a cancellation leaves the survivors' stamps alone ------------------------------
    SpecQuote(
        "market-spec.md", "§5.2",
        "Each order receives a timestamp when entering the SOB.",
        note="Entry order is timestamp order.",
    ),
    SpecQuote(
        "market-spec.md", "§5.2",
        "A **new timestamp** is assigned on: modification (price, quantity, execution restriction), reactivation "
        "from inactive, or activation of a new iceberg slice.",
        note="Another order's cancellation is not on this list: every survivor keeps its stamp, so its rank.",
    ),
    SpecQuote(
        "market-spec.md", "§5.2",
        "All timestamps across orders and explicit capacity requests are assigned from a single global queue "
        "(unique, sequential). Source: Continuous Trading Matching Algorithm §4.3.",
        note="Asserted on the survivors' stored stamps after the cancel: unique, strictly increasing in entry order.",
    ),
    # --- §6.2: the aggressor's walk after the cancel ----------------------------------------------
    SpecQuote(
        "market-spec.md", "§6.2 item 1",
        "Incoming order attempts to match against best counterpart in the consolidated order book.",
        note="The aggressor (ladder) and the FOK (iceberg) are the incoming orders; 'best' is §6.1's ranking.",
    ),
    SpecQuote(
        "market-spec.md", "§6.2 item 2",
        "Match is possible if: orders are opposite sides, same contract, and `sell_price ≤ buy_price`.",
        note="Opposite sides are required: a cancel among same-side rows cannot produce a match.",
    ),
    SpecQuote(
        "market-spec.md", "§6.2 item 3",
        "Trade executes at the **resting order's limit price** (maker pricing).",
        note="Every ladder fill prints at the touch price; every control fill at the iceberg's price.",
    ),
    # --- §4.5 / §4.2: who rests, what is shown, what a FOK may do -----------------------------------
    SpecQuote(
        "market-spec.md", "§4.5",
        "| Restriction | Partial fill | Book entry if unmatched | Applicable to |",
        note="Header of the execution-restriction table, quoted so the rows below can be read.",
    ),
    SpecQuote(
        "market-spec.md", "§4.5",
        "| NON | Yes | Yes | Regular, Block |",
        note="Touch orders, decoys and visible orders rest; an untouched one stays in full.",
    ),
    SpecQuote(
        "market-spec.md", "§4.5",
        "| FOK | No (full or cancel) | No (cancelled) | Regular, Linked basket |",
        note="The FOK fills all of R or nothing, and never enters the book.",
    ),
    SpecQuote(
        "market-spec.md", "§4.5",
        "| ICE | Yes (peak slices) | Yes (hidden qty rests) | Iceberg only |",
        note="Before the cancel the iceberg rests with its hidden quantity.",
    ),
    SpecQuote(
        "market-spec.md", "§4.2",
        "Only the peak size is shown in the order book.",
        note="Before the cancel: visible P, hidden T - P.",
    ),
    # --- §4.2.1 Rules 1-2: what the FOK pre-check counts -------------------------------------------
    SpecQuote(
        "market-spec.md", "§4.2.1",
        "A Fill-or-Kill order's pre-match liquidity check considers the total iceberg quantity for opposite-side "
        "icebergs at crossing prices. For **standard icebergs (PPD=0)**, all remaining quantity is matchable if "
        "the base price crosses.",
        note="Rule 1. A cancelled iceberg has no remaining quantity; uncancelled (the control), all of T counts.",
    ),
    SpecQuote(
        "market-spec.md", "§4.2.1",
        "**Standard iceberg example:** FOK BUY 3 MW is accepted against an iceberg with peak=2 and total=7 "
        "(PPD=0), because all 7 MW are at the same crossing price.",
        note="The control's shape: a FOK larger than the peak is accepted against the iceberg's total.",
    ),
    SpecQuote(
        "market-spec.md", "§4.2.1",
        "A single aggressive order (NON, IOC, or FOK) can fill multiple iceberg slices within one match cycle.",
        note="Rule 2: in the control the FOK takes as many slices as it needs in its one submission.",
    ),
    # --- §5.3: the hard rule left uncertified by scope decision -------------------------------------
    SpecQuote(
        "market-spec.md", "§5.3",
        "Modifying price, quantity, or execution restriction of an order resets its timestamp (loses time "
        "priority).",
        note="NOT certified here: excluded by scope decision, though implemented (modify_order).",
    ),
    SpecQuote(
        "market-spec.md", "§5.3",
        "This is a hard rule from the XBID system.",
        note="The one XBID hard rule C04 leaves uncertified.",
    ),
    # --- the stale-slot hazard ------------------------------------------------------------------------
    SpecQuote(
        "open-questions.md", "OQ-20",
        "A slot whose OID is absent from the LOB but whose `remaining_hidden > 0` would either over-report "
        "matchable hidden volume (if the meta row's quantity fields are still valid and the next slice has yet "
        "to activate) or under-report (if the slot represents a fully-consumed iceberg that was not cleaned up). "
        "Both are correctness-affecting silent bugs at the FOK pre-check path",
        block="Why the test is still load-bearing",
        note="The hazard the iceberg draw targets: a stale side-channel row over-reporting to the FOK pre-check.",
    ),
    SpecQuote(
        "scenario-ladder-review.md", "§2",
        "a cancelled iceberg's hidden remainder must be invisible to a later FOK pre-check, which reads "
        "`iceberg_meta` directly.",
        note="Why C04 was added: the review's statement of the iceberg claim.",
    ),
)

_MA = ("docs/omiecid/corpus/Mapping_Authoritative_Documents_EU_Continuous_Intraday_Trading_SIDC_XBID_Spain_OMIE/"
       "Continuous_Trading_Matching_Algorithm_Public_Description.md")
CITATIONS = (
    Citation(
        "Continuous Trading Matching Algorithm (public description) §4.1 Consolidated order books — " + _MA,
        "The order book is updated whenever orders are added to it, deleted from it, modified (volume, price, "
        "execution restriction, activation of a new iceberg slice) , or matched (either partially or fully), or "
        "the underlying contract expires or trading on it is halted. [...] After each order book update or local "
        "view update, the algorithm validates if new matching opportunities have arisen and, if so, the resulting "
        "trades are concluded.",
    ),
    Citation(
        "Continuous Trading Matching Algorithm (public description) §4.1 — same file",
        "Entry into the order book means that the order is stored in the system and made visible to other users "
        "until either the order is deleted or deactivated or a matching order is submitted.",
    ),
    Citation(
        "Continuous Trading Matching Algorithm (public description) §3.5.3 FOK — same file",
        "An order submitted with the execution restriction FOK is either executed immediately and with its full "
        "quantity or, if the order cannot be matched with its entire quantity, deleted without entry in the order "
        "book.",
    ),
)

QUOTE_CHECKS = check_quotes(QUOTES)
display(Markdown(render_quotes(QUOTE_CHECKS, CITATIONS)))
print(f"quotes: {sum(c.ok for c in QUOTE_CHECKS)}/{len(QUOTE_CHECKS)} match their source")

## 3. Configuration

One visible parameter cell. The headless gate overrides it by injecting a cell right after it (`execute_notebook(..., parameters={...})`); everything derived from it is computed in the next cell.

- **`DRAWS = 500`** valid draws of `C04.cancellation` and **`ICE_DRAWS = 500`** of `C04.iceberg_cancel_fok` (the card's default for a random scenario, per construction). The run requests `DRAWS + ICE_DRAWS` valid draws.
- **`SEED0 = 0`**, the first seed of each construction. A draw is **discarded and replaced by the next seed** if it is found outside the capacity margin before the run, or if the battery marks its main run I8 `invalid` afterwards. Each construction's loop stops at its draw count or after twice that many attempts, and the discard rate is reported.
- **`EAGER_DRAWS = 5`.** The first `EAGER_DRAWS` **valid** draws of each construction also run eager, and are compared integer field by integer field with their jitted run (`compare_records`). Further eager runs are **forced**: the first valid run of every class not yet covered. The classes are: ladder draw × resting side; `decoys_first` × resting side; the two sweeps; iceberg draw × resting side × ($n_{vis} = 0$ or $\ge 1$); and the no-cancel control × resting side. Every one of these **must** have its eager comparison, and a missing one is recorded as a failed `eager_jit` assertion. Eager runs are chosen among valid draws, so a discarded seed cannot remove them.
- **`SWEEP_DRAWS = 20`.** The first `SWEEP_DRAWS` valid ladder draws also run the victim sweep: each of the $k$ touch orders cancelled in turn, on the drawn tape and on `decoys_first`.
- The `decoys_first` run and the no-cancel control run on **every** valid draw of their construction.
- The tick grids are **not** parameters: they come from `OMIEProductConfig()` and are used only to display euros. Every assertion is in integer ticks and order ids.

A run with far fewer draws than the default (say, under 30) can miss a value of $k$, of the decoy count, of the victim's position or of $n_{vis}$, or a class. It then fails the coverage checks and the missing class's eager comparison. That is intended: such a run does not cover the card's dataset.

In [ ]:
CERT_ID = "C04"        # C00-C17 are the ladder; C90-C99 are reserved for the harness
DRAWS = 500            # valid draws of C04.cancellation (card: random scenarios default to 500 draws)
ICE_DRAWS = 500        # valid draws of C04.iceberg_cancel_fok (the card's separate iceberg draw)
SEED0 = 0              # first seed of each construction
EAGER_DRAWS = 5        # the first EAGER_DRAWS valid draws of each construction also run eager (plus forced classes)
SWEEP_DRAWS = 20       # the first SWEEP_DRAWS valid ladder draws also run the victim sweep

In [ ]:
import pathlib

import numpy as np

from omiecid_simulator.jaxomiecid import omie_tape_synth as synth
from omiecid_simulator.omiecid.omie_config import OMIEProductConfig
from omiecid_simulator.omiecid.order_types import ExecRestriction, Side

LADDER = "C04.cancellation"
ICEBERG = "C04.iceberg_cancel_fok"
CONSTRUCTIONS = (LADDER, ICEBERG)
N_DRAWS = {LADDER: DRAWS, ICEBERG: ICE_DRAWS}
REQUESTED = DRAWS + ICE_DRAWS

CONFIG = OMIEProductConfig()
ENVELOPE = synth.DEFAULT_ENVELOPE
BUY, SELL = int(Side.BUY), int(Side.SELL)
NON, FOK, ICE = int(ExecRestriction.NON), int(ExecRestriction.FOK), int(ExecRestriction.ICE)

SLICE_MARGIN = 64             # capacity-validity precondition: every iceberg has ceil(total/peak) <= 64
PRICE_INT_MAX = 1_999_800     # §2.3: +9,999 EUR/MWh in the offset encoding (0 is -9,999)
QTY_INT_MIN, QTY_INT_MAX = 1, 9_990   # §2.3: 0.1 MW and 999 MW in quantity ticks
K_RANGE = tuple(range(2, 6))  # card: k in [2, 5] touch orders
DECOY_RANGE = (1, 2, 3)       # the generator's decoy count
NVIS_RANGE = (0, 1, 2, 3)     # the generator's visible-order count beside the iceberg

_own = sorted(pathlib.Path.cwd().glob(f"{CERT_ID}_*.ipynb"))
NOTEBOOK = _own[0].name if len(_own) == 1 else "C04_cancellation.ipynb"

print(f"{CERT_ID}: {LADDER} x {DRAWS}, {ICEBERG} x {ICE_DRAWS} valid draws (requested {REQUESTED}), "
      f"seeds from {SEED0}, at most 2x attempts per construction")
print(f"eager: first {EAGER_DRAWS} valid draws per construction + forced classes; victim sweep on the first "
      f"{SWEEP_DRAWS} ladder draws; decoys_first and the no-cancel control on every draw")
print(f"price tick {CONFIG.price_tick} EUR/MWh, qty tick {CONFIG.qty_tick} MW (display only)")
print(f"capacity envelope: {ENVELOPE}; slice margin <= {SLICE_MARGIN} per iceberg")

## 4. Dataset

**`C04.cancellation`** (`omie_tape_synth.make_cancellation`, `variant="ladder"`) draws, per seed: $k \in [2,5]$, $j \in [1, k-1]$, a decoy count in $[1,3]$, the resting side (BUY or SELL with probability ½), a touch price on the grid in [30, 90] EUR/MWh, one size $q \in [20, 200]$ ticks (2–20 MW), the victim's position among the touch orders, and two steps in $[1,50]$ ticks. All rows are NON on one contract. Order ids and tape timestamps are $1, 2, 3, \dots$ in row order. The tape holds **the $k$ touch orders** (agent 1) at the touch price, each for $q$. Next come **the decoys** (agent 1), each for $q$: the $d$-th sits $d$ steps *beyond the aggressor's limit*, so no decoy can trade. Last comes **the aggressor** (agent 2) on the other side, priced through the touch, for $j \cdot q$. The victim (`params["cancel_oid"]`) is cancelled immediately before the aggressor's row.

**`C04.iceberg_cancel_fok`** (`variant="iceberg_fok"`) draws the resting side, a price on the grid in [30, 90] EUR/MWh, a peak $P \in [10, 80]$ ticks and a slice count $n \in [2, 20]$, with total $T = (n-1)P + r$, $1 \le r \le P$. It also draws $n_{vis} \in [0,3]$ visible sizes in $[10, 200]$ ticks, summing to $V_{vis}$, and a FOK size $R \in [V_{vis}+1,\ V_{vis}+T]$. The tape holds **the iceberg** (agent 1, ICE, PPD 0), then **the visible NON orders** (agent 3) at its price, then **the FOK** (agent 2) on the other side, priced through. The iceberg is cancelled immediately before the FOK's row.

**The cancellation plan.** For both constructions the plan is `{order id of the last real row: (params["cancel_oid"],)}`, which is `build_record`'s `cancels_before`. The precondition checks that `draw.meta["cancel_before_row_of_oid"]` names that same row.

**What section 5 reads.** The expectation reads the **tape rows** (their order, sides, prices, quantities, order ids, restrictions and peaks) and the cancellation plan. `draw.params` is used for display, for the closed-form cross-check of section 5 and for a consistency check against the tape. `draw.expected` is never read.

**The precondition, checked from the tape for every valid draw** (an aggregate dataset assertion in section 7).
- *Ladder:* every row NON; one contract; prices and quantities within §2.3's bounds; distinct order ids; tape timestamps strictly increasing; every row but the last on one side and the last on the other; $k \in [2,5]$ rows at the best resting price, all of size $q$; 1 to 3 decoys, each strictly worse than the touch **and** not crossing the aggressor; every decoy entered after every touch order; the aggressor crossing the touch, sized $j \cdot q$ with $1 \le j \le k-1$; the victim a touch order; the plan before the aggressor's row; and `draw.params` equal to the tape's integers.
- *Iceberg:* the first row ICE, PPD 0, peak $0 < P < T$, with $\lceil T/P \rceil \in [2, 64]$; then $n_{vis} \in [0,3]$ NON rows on the same side at the same price; then one FOK row on the other side crossing that price; $V_{vis} < R \le V_{vis} + T$; the victim the iceberg; the plan before the FOK's row; and `draw.params` equal to the tape's integers.

**The capacity margin** (scenario-ladder-plan, *The capacity-validity precondition*, five ceilings). These tapes sit far inside it: at most 9 orders, at most one iceberg of at most 20 slices. It is still checked **before** each run, on draws and derived runs alike. A tape is never run if its analytic bounds exceed the envelope (≤ 80 resting per side, ≤ 80 trades, hop bound ≤ 160, ≤ 800 icebergs) or if its largest $\lceil qty/peak \rceil$ exceeds 64. A draw that fails is discarded and re-drawn; a derived run that fails is recorded as a failed check. The largest slice count is reported. After each run, the battery's I8 confirms the flags.

**Derived runs.** Each changes only what its row says. $T_1, \dots, T_k$ are the touch orders in drawn entry order.

| run | made from | change | on |
|---|---|---|---|
| `decoys_first` | ladder draw | the decoy rows re-entered **before** the touch rows (decoys, then $T_1..T_k$, then the aggressor). Order ids travel with their rows; the timestamp column keeps its values by slot, so it still grows down the tape. Same victim | every draw |
| `sweep[Ti]` | ladder draw | the victim replaced by $T_i$, for $i = 1..k$ | first `SWEEP_DRAWS` |
| `sweep_df[Ti]` | `decoys_first` | the victim replaced by $T_i$, for $i = 1..k$ | first `SWEEP_DRAWS` |
| `no_cancel` | iceberg draw | the cancellation plan emptied: the iceberg is never cancelled | every draw |

Below: the first seed of each construction in readable units beside the integers, a per-construction summary, the first ladder seed's `decoys_first` tape, and plots of the book shape and of the dataset's coverage.

In [ ]:
from collections import Counter

from omiecid_simulator.omiecid.encoding import decode_price, decode_qty

C_SIDE, C_PRICE, C_QTY, C_OID = synth.COL_SIDE, synth.COL_PRICE, synth.COL_QTY, synth.COL_OID
C_AGENT, C_TS, C_EXEC, C_CONTRACT = synth.COL_AGENT, synth.COL_TS, synth.COL_EXEC, synth.COL_CONTRACT
C_PEAK, C_PPD = synth.COL_PEAK, synth.COL_PPD


def real_rows(tape):
    """The real rows (order_id >= 1), in tape (= submission) order."""
    t = np.asarray(tape)
    return t[t[:, C_OID] >= 1]


def real_index(tape):
    """Tape row indices of the real rows, in order."""
    return [int(i) for i in np.nonzero(np.asarray(tape)[:, C_OID] >= 1)[0]]


def side_name(side):
    return "BUY" if int(side) == BUY else "SELL"


def better(rest_side, p, p_ref):
    """True when resting price p ranks strictly ahead of p_ref (§6.1)."""
    return int(p) < int(p_ref) if int(rest_side) == SELL else int(p) > int(p_ref)


def crosses(rest_side, p_rest, p_agg):
    """§6.1 / §6.2 item 2: sell price <= buy price."""
    return int(p_rest) <= int(p_agg) if int(rest_side) == SELL else int(p_rest) >= int(p_agg)


def ceil_div(a, b):
    return -(-int(a) // int(b))


def max_slices(tape):
    """Largest ceil(qty / peak) over the tape's iceberg rows (0 when there are none)."""
    ice = [r for r in real_rows(tape) if int(r[C_EXEC]) == ICE]
    return max((ceil_div(r[C_QTY], max(1, int(r[C_PEAK]))) for r in ice), default=0)


def bounds_ok(tape):
    return all(0 <= int(r[C_PRICE]) <= PRICE_INT_MAX and QTY_INT_MIN <= int(r[C_QTY]) <= QTY_INT_MAX
               for r in real_rows(tape))


def plan_of(draw):
    """The cancellation plan: {order id of the last real row: (params['cancel_oid'],)}."""
    rows = real_rows(draw.tape)
    return {int(rows[-1][C_OID]): (int(draw.params["cancel_oid"]),)}


def shape(tape):
    """The tape read as 'resting rows in entry order, then one incoming order', with the touch level.

    Returns None when the rows before the last are not all on the side opposite the last row."""
    rows = real_rows(tape)
    if len(rows) < 2:
        return None
    rest, agg = rows[:-1], rows[-1]
    rs = int(rest[0][C_SIDE])
    if any(int(r[C_SIDE]) != rs for r in rest) or int(agg[C_SIDE]) == rs:
        return None
    best = rest[0][C_PRICE]
    for r in rest:
        if better(rs, r[C_PRICE], best):
            best = r[C_PRICE]
    touch = [int(r[C_OID]) for r in rest if int(r[C_PRICE]) == int(best)]
    decoys = [int(r[C_OID]) for r in rest if int(r[C_PRICE]) != int(best)]
    pos = {int(r[C_OID]): i for i, r in enumerate(rest)}
    by_oid = {int(r[C_OID]): r for r in rows}
    return {"rest": rest, "agg": agg, "rest_side": rs, "touch": touch, "decoys": decoys, "pos": pos,
            "by_oid": by_oid, "p_touch": int(best)}


def params_diff(p, want):
    return {kk: (p.get(kk), v) for kk, v in want.items() if p.get(kk) != v}


def precondition_ladder(draw):
    """Every way a ladder draw departs from the dataset described above (empty list = as described)."""
    tape, p, bad = draw.tape, draw.params, []
    rows = real_rows(tape)
    sh = shape(tape)
    if sh is None:
        return ["not 'resting rows on one side, then one aggressor on the other'"]
    if any(int(r[C_EXEC]) != NON for r in rows):
        bad.append("a row is not NON")
    if len({int(r[C_CONTRACT]) for r in rows}) != 1:
        bad.append("more than one contract")
    if not bounds_ok(tape):
        bad.append("a row outside §2.3's bounds")
    if len({int(r[C_OID]) for r in rows}) != len(rows):
        bad.append("duplicate order ids")
    if not all(int(a[C_TS]) < int(b[C_TS]) for a, b in zip(rows[:-1], rows[1:])):
        bad.append("tape timestamps not strictly increasing down the tape")
    touch, decoys, agg, rs = sh["touch"], sh["decoys"], sh["agg"], sh["rest_side"]
    q = {int(sh["by_oid"][o][C_QTY]) for o in touch}
    k = len(touch)
    if k not in K_RANGE:
        bad.append(f"k = {k} touch orders (expected 2..5)")
    if len(q) != 1:
        bad.append(f"touch sizes differ: {sorted(q)}")
    if len(decoys) not in DECOY_RANGE:
        bad.append(f"{len(decoys)} decoys (expected 1..3)")
    if any(crosses(rs, sh["by_oid"][d][C_PRICE], agg[C_PRICE]) for d in decoys):
        bad.append("a decoy crosses the aggressor's limit (it should be unreachable)")
    if not crosses(rs, sh["p_touch"], agg[C_PRICE]):
        bad.append("the aggressor does not cross the touch")
    q_each = min(q) if q else 0
    q_agg = int(agg[C_QTY])
    j = q_agg // q_each if q_each else -1
    if not q_each or q_agg != j * q_each or not 1 <= j <= k - 1:
        bad.append(f"aggressor size {q_agg} is not j * q with 1 <= j <= k - 1 (q = {q_each}, k = {k})")
    if not all(sh["pos"][d] > max(sh["pos"][o] for o in touch) for d in decoys):
        bad.append("a decoy entered before a touch order")
    victim = int(p.get("cancel_oid", -1))
    if victim not in touch:
        bad.append(f"the victim {victim} is not a touch order")
    if int(draw.meta.get("cancel_before_row_of_oid", -1)) != int(agg[C_OID]):
        bad.append("the cancellation is not scheduled before the aggressor's row")
    want = {"k": k, "j": j, "n_decoys": len(decoys), "rest_side": rs, "agg_side": int(agg[C_SIDE]),
            "p_touch_int": sh["p_touch"], "q_each_int": q_each, "resting_oids": touch, "decoy_oids": decoys,
            "agg_oid": int(agg[C_OID]), "p_agg_int": int(agg[C_PRICE]), "q_agg_int": q_agg}
    diff = params_diff(p, want)
    if diff:
        bad.append(f"params disagree with the tape (params, tape): {diff}")
    return bad


def ice_shape(tape):
    """The iceberg tape read as: iceberg, visible orders, FOK. None when it is not that shape."""
    rows = real_rows(tape)
    if len(rows) < 2:
        return None
    ice, vis, fok = rows[0], rows[1:-1], rows[-1]
    return {"ice": ice, "vis": vis, "fok": fok, "rest_side": int(ice[C_SIDE]), "ice_oid": int(ice[C_OID]),
            "vis_oids": [int(r[C_OID]) for r in vis], "fok_oid": int(fok[C_OID]), "p_ice": int(ice[C_PRICE]),
            "total": int(ice[C_QTY]), "peak": int(ice[C_PEAK]), "v_vis": int(sum(int(r[C_QTY]) for r in vis)),
            "req": int(fok[C_QTY]), "p_fok": int(fok[C_PRICE])}


def precondition_iceberg(draw):
    """Every way an iceberg draw departs from the dataset described above."""
    tape, p, bad = draw.tape, draw.params, []
    rows = real_rows(tape)
    s = ice_shape(tape)
    if s is None:
        return [f"{len(rows)} real rows (expected the iceberg, 0-3 visible orders and a FOK)"]
    ice, fok, rs = s["ice"], s["fok"], s["rest_side"]
    if int(ice[C_EXEC]) != ICE or int(ice[C_PPD]) != 0:
        bad.append("the first row is not a standard (PPD 0) iceberg")
    if not 0 < s["peak"] < s["total"]:
        bad.append(f"peak {s['peak']} not in (0, total {s['total']})")
    n_sl = ceil_div(s["total"], max(1, s["peak"]))
    if not 2 <= n_sl <= SLICE_MARGIN:
        bad.append(f"ceil(total/peak) = {n_sl} not in [2, {SLICE_MARGIN}]")
    if len(s["vis"]) not in NVIS_RANGE:
        bad.append(f"{len(s['vis'])} visible orders (expected 0..3)")
    for r in s["vis"]:
        if int(r[C_EXEC]) != NON or int(r[C_SIDE]) != rs or int(r[C_PRICE]) != s["p_ice"]:
            bad.append(f"visible order {int(r[C_OID])} is not NON on the iceberg's side at its price")
    if int(fok[C_EXEC]) != FOK or int(fok[C_SIDE]) == rs:
        bad.append("the last row is not a FOK on the other side")
    if not crosses(rs, s["p_ice"], s["p_fok"]):
        bad.append("the FOK does not cross the iceberg's price")
    if not s["v_vis"] < s["req"] <= s["v_vis"] + s["total"]:
        bad.append(f"R = {s['req']} not in (V_vis = {s['v_vis']}, V_vis + T = {s['v_vis'] + s['total']}]")
    if len({int(r[C_CONTRACT]) for r in rows}) != 1:
        bad.append("more than one contract")
    if not bounds_ok(tape):
        bad.append("a row outside §2.3's bounds")
    if len({int(r[C_OID]) for r in rows}) != len(rows):
        bad.append("duplicate order ids")
    if not all(int(a[C_TS]) < int(b[C_TS]) for a, b in zip(rows[:-1], rows[1:])):
        bad.append("tape timestamps not strictly increasing down the tape")
    if int(p.get("cancel_oid", -1)) != s["ice_oid"]:
        bad.append("the victim is not the iceberg")
    if int(draw.meta.get("cancel_before_row_of_oid", -1)) != s["fok_oid"]:
        bad.append("the cancellation is not scheduled before the FOK's row")
    want = {"rest_side": rs, "ice_oid": s["ice_oid"], "p_ice_int": s["p_ice"], "total_int": s["total"],
            "peak_int": s["peak"], "n_slices": n_sl, "n_vis": len(s["vis"]), "visible_oids": s["vis_oids"],
            "v_vis_int": s["v_vis"], "fok_oid": s["fok_oid"], "req_int": s["req"], "p_fok_int": s["p_fok"]}
    diff = params_diff(p, want)
    if diff:
        bad.append(f"params disagree with the tape (params, tape): {diff}")
    return bad


def decoys_first(tape):
    """The decoy rows re-entered before the touch rows; the timestamp column kept by slot."""
    base = np.array(tape, copy=True)
    idx = real_index(base)
    sh = shape(base)
    slot = {int(base[i, C_OID]): i for i in idx}
    rest_slots = idx[:-1]
    order = [slot[o] for o in sh["decoys"]] + [slot[o] for o in sh["touch"]]
    t = base.copy()
    t[rest_slots] = base[order]
    t[:, C_TS] = base[:, C_TS]          # the timestamp column keeps its values by slot
    return np.ascontiguousarray(t, dtype=np.int32)


DRAWN, DF_TAPE = {}, {}


def prepare(key, seed):
    """Draw seed `seed` of construction `key` (initial seeds and re-draws), with its derived tape."""
    d = synth.CONSTRUCTIONS[key](seed)
    DRAWN[(key, seed)] = d
    if key == LADDER and shape(d.tape) is not None:
        DF_TAPE[seed] = decoys_first(d.tape)
    return d


for key in CONSTRUCTIONS:
    for s in range(SEED0, SEED0 + N_DRAWS[key]):
        prepare(key, s)


def rows_table(tape, title, victim=None):
    rows = real_rows(tape)
    role = {}
    sh = shape(tape)
    if sh is not None:
        if any(int(r[C_EXEC]) == ICE for r in rows):
            for r in rows[:-1]:
                role[int(r[C_OID])] = "iceberg" if int(r[C_EXEC]) == ICE else "visible"
            role[int(sh["agg"][C_OID])] = "FOK"
        else:
            role.update({o: f"touch T{n}" for n, o in enumerate(sh["touch"], start=1)})
            role.update({o: "decoy" for o in sh["decoys"]})
            role[int(sh["agg"][C_OID])] = "aggressor"
    if victim is not None:
        role[int(victim)] = role.get(int(victim), "") + " **(cancelled)**"
    lines = [f"**{title}**", "",
             "| row | role | oid | side | exec | price (EUR/MWh) | qty (MW) | peak (MW) | price_int | qty_int | peak_int | agent | ts |",
             "|---|---|---|---|---|---|---|---|---|---|---|---|---|"]
    ex = {NON: "NON", FOK: "FOK", ICE: "ICE"}
    for n, r in enumerate(rows, start=1):
        pk = int(r[C_PEAK])
        lines.append(
            f"| {n} | {role.get(int(r[C_OID]), '')} | {int(r[C_OID])} | {side_name(r[C_SIDE])} | "
            f"{ex.get(int(r[C_EXEC]), int(r[C_EXEC]))} | {decode_price(int(r[C_PRICE]), CONFIG):.2f} | "
            f"{decode_qty(int(r[C_QTY]), CONFIG):.1f} | {decode_qty(pk, CONFIG):.1f} | {int(r[C_PRICE])} | "
            f"{int(r[C_QTY])} | {pk} | {int(r[C_AGENT])} | {int(r[C_TS])} |")
    return "\n".join(lines)


for key in CONSTRUCTIONS:
    d = DRAWN[(key, SEED0)]
    display(Markdown(rows_table(d.tape, f"{key}, seed {SEED0} (digest `{d.digest()[:16]}…`, tape {d.tape.shape}, "
                                        f"hop bound {d.hop_bound}); cancellation plan {plan_of(d)}",
                                victim=d.params["cancel_oid"])))
display(Markdown(rows_table(DF_TAPE[SEED0], f"{LADDER}, seed {SEED0}, derived run `decoys_first`",
                            victim=DRAWN[(LADDER, SEED0)].params["cancel_oid"])))

_lp = [DRAWN[(LADDER, s)].params for s in range(SEED0, SEED0 + DRAWS)]
print(f"{LADDER}: {len(_lp)} seeds; k {dict(sorted(Counter(p['k'] for p in _lp).items()))}; "
      f"j = 1: {sum(p['j'] == 1 for p in _lp)}, j = k - 1: {sum(p['j'] == p['k'] - 1 for p in _lp)}; "
      f"decoys {dict(sorted(Counter(p['n_decoys'] for p in _lp).items()))}; "
      f"victim position {dict(sorted(Counter(p['resting_oids'].index(p['cancel_oid']) + 1 for p in _lp).items()))}; "
      f"resting SELL {sum(p['rest_side'] == SELL for p in _lp)}, BUY {sum(p['rest_side'] == BUY for p in _lp)}")
_ip = [DRAWN[(ICEBERG, s)].params for s in range(SEED0, SEED0 + ICE_DRAWS)]
print(f"{ICEBERG}: {len(_ip)} seeds; n_vis {dict(sorted(Counter(p['n_vis'] for p in _ip).items()))}; "
      f"slices {min(p['n_slices'] for p in _ip)}..{max(p['n_slices'] for p in _ip)}; "
      f"R > V_vis + P (only hidden volume could satisfy): {sum(p['req_int'] > p['v_vis_int'] + p['peak_int'] for p in _ip)}; "
      f"that class with n_vis >= 1: {sum(p['req_int'] > p['v_vis_int'] + p['peak_int'] and p['n_vis'] >= 1 for p in _ip)}; "
      f"resting SELL {sum(p['rest_side'] == SELL for p in _ip)}, BUY {sum(p['rest_side'] == BUY for p in _ip)}")
print("largest analytic bounds over all seeds: "
      + ", ".join(f"{a} {max(d.capacity_bounds[a] for d in DRAWN.values())}" for a in synth.AXES)
      + f"; largest ceil(qty/peak) {max(max_slices(d.tape) for d in DRAWN.values())}")

In [ ]:
import matplotlib.pyplot as plt

ROLE_COLOUR = {"touch": "tab:blue", "decoy": "tab:grey", "aggressor": "tab:orange", "victim": "tab:red",
               "iceberg": "tab:purple", "visible": "tab:blue", "FOK": "tab:orange"}
fig, axes = plt.subplots(1, 4, figsize=(18, 3.6))


def book_plot(ax, tape, victim, title):
    sh = shape(tape)
    rows = real_rows(tape)
    is_ice = any(int(r[C_EXEC]) == ICE for r in rows)
    for n, r in enumerate(rows):
        o = int(r[C_OID])
        if o == int(victim):
            role = "victim"
        elif o == int(sh["agg"][C_OID]):
            role = "FOK" if is_ice else "aggressor"
        elif is_ice:
            role = "iceberg" if int(r[C_EXEC]) == ICE else "visible"
        else:
            role = "touch" if o in sh["touch"] else "decoy"
        sign = 1 if int(r[C_SIDE]) == BUY else -1
        y = decode_price(int(r[C_PRICE]), CONFIG) + 0.03 * n     # stagger same-price rows so each is visible
        ax.barh(y, decode_qty(int(r[C_QTY]), CONFIG) * sign, height=0.025, color=ROLE_COLOUR[role],
                hatch="//" if role == "victim" else None)
        ax.annotate(f"#{o}", (decode_qty(int(r[C_QTY]), CONFIG) * sign, y), fontsize=6)
    ax.axvline(0, color="black", lw=0.8)
    ax.set_xlabel("quantity (MW): buy right, sell left")
    ax.set_ylabel("limit price (EUR/MWh), rows staggered")
    ax.set_title(title, fontsize=8)


_d0 = DRAWN[(LADDER, SEED0)]
book_plot(axes[0], _d0.tape, _d0.params["cancel_oid"],
          f"ladder, seed {SEED0}\n(blue touch, red hatched victim, grey decoy, orange aggressor)")
_i0 = DRAWN[(ICEBERG, next(s for s in range(SEED0, SEED0 + ICE_DRAWS) if DRAWN[(ICEBERG, s)].params["n_vis"] >= 1))]
book_plot(axes[1], _i0.tape, _i0.params["cancel_oid"],
          f"iceberg, seed {_i0.seed} (first with n_vis >= 1)\n(red hatched: the cancelled iceberg's TOTAL; blue visible; orange FOK)")

kv = Counter((p["k"], p["resting_oids"].index(p["cancel_oid"]) + 1) for p in _lp)
axes[2].scatter([a for a, _ in kv], [b for _, b in kv], s=[4 * n for n in kv.values()], alpha=0.6)
axes[2].set_xlabel("k (touch orders)")
axes[2].set_ylabel("victim position i (T_i)")
axes[2].set_title("ladder: (k, victim) coverage; marker area = draws", fontsize=8)

for nv in NVIS_RANGE:
    xs = [p["req_int"] - p["v_vis_int"] for p in _ip if p["n_vis"] == nv]
    ys = [p["total_int"] for p in _ip if p["n_vis"] == nv]
    axes[3].scatter(xs, ys, s=6, alpha=0.6, label=f"n_vis = {nv}")
_mx = max(p["total_int"] for p in _ip)
axes[3].plot([0, _mx], [0, _mx], color="black", lw=0.8)
axes[3].set_xlabel("R - V_vis (ticks): what only the iceberg could supply")
axes[3].set_ylabel("T (ticks)")
axes[3].set_title("iceberg: 0 < R - V_vis <= T (on or left of the diagonal)", fontsize=8)
axes[3].legend(fontsize=6)
plt.tight_layout()
plt.show()

## 5. Known result

**Notation.** Integer ticks throughout. On a ladder tape the real rows, in submission order, are the resting rows $r_1, \dots, r_m$ (one side, $\sigma$), then the aggressor $a$ on the other side, with limit $p_a$, size $q_a$ and order id $o_a$. Row $r_i$ has limit $p_i$, size $q_i$, order id $o_i$ and **entry index** $i$. The cancellation plan removes a set $C$ (here one order, the victim) just before $a$ is submitted.

**(X) The cancellation** (§5.1, §9.1; §6.1, §6.2 items 1–2). Each $r_i$ meets no counterpart on entry and rests in full (§4.5). The victim is active in the book, so the cancel is valid, and its effect is removal: the victim's fate is `cancellation` for its full size $q_v$, it never fills, and no other row changes. Every resting row is on one side, so no pair crosses and the cancel prints **no trade**. The **survivors** are $S = (r_i : o_i \notin C)$, kept in entry order.

**(R) Ranking after the removal** (§5.2, §6.1). A survivor's timestamp was given on entry. A cancellation of another order is not one of §5.2's new-timestamp events, so the survivor keeps its stamp. The survivors are ranked by
$$\kappa(r_i) = \begin{cases} (p_i,\ i) & \sigma = \text{SELL}\\ (-p_i,\ i) & \sigma = \text{BUY} \end{cases}$$
in increasing lexicographic order. Their stored stamps, listed in entry order, are strictly increasing.

**(W) The walk** (§6.2 items 1, 3; §6.1's $\ge$). Start with remainder $\rho = q_a$. Take the survivors in $\kappa$ order. While $\rho > 0$ and the next one crosses, fill $f_i = \min(\rho, q_i)$ at $p_i$ and set $\rho \leftarrow \rho - f_i$. Stop at the first survivor that does not cross.

**What is asserted, per ladder run:**
- **(C1) removal**: the book just before $a$ equals, row for row (price, quantity, order id, owner, stored stamp), the book just after $r_m$ with the victim's row taken out. The victim holds no visible or hidden quantity. Its fate is exactly one `cancellation` of $q_v$, and it fills $0$;
- **(C2) no trade from the cancel**: the trade tape just before $a$ is byte-identical to the one just after $r_m$;
- **(T) stored stamps**: in the book just before $a$, each survivor holds exactly one entry, and the stamps $(\text{sec}, \text{nsec})$ listed in entry order are strictly increasing;
- **(B0) book before the aggressor** $=$ every survivor at $(p_i, q_i)$;
- **(S) filled set** $= \{o_i : f_i > 0\}$ and **(Q) counterparty sequence** $= [o_i,\ o_a,\ p_i,\ f_i,\ \text{on } a\text{'s submission}]$ in walk order;
- **(B) book afterwards** $=$ every survivor with $q_i - f_i > 0$ at $(p_i, q_i - f_i)$, plus $a$ at $(p_a, \rho)$ when $\rho > 0$. The victim is absent.

**Closed form on a ladder draw** (from `draw.params`, cross-checked against (W) on every draw, on its `decoys_first` run and on every sweep run). The touch orders $T_1..T_k$ are in entry order, each of size $q$ at $p_{touch}$; the victim is $T_v$; and $q_a = j \cdot q$ with $1 \le j \le k-1$. Write $S = (T_1, \dots, T_k) \setminus T_v$, which has $k-1$ orders. Then
$$\text{filled} = \{S_1, \dots, S_j\},\qquad \text{sequence} = [(S_1, q), \dots, (S_j, q)] \text{ at } p_{touch},\qquad \text{rest} = \{S_{j+1}, \dots, S_{k-1}\} \cup \text{decoys, all at full size}.$$
The aggressor does not rest. The decoys lie beyond its limit, so they never trade, wherever they were entered. In `decoys_first` the decoys come first in entry order, but they are worse-priced, so $S$ and the result are unchanged. `sweep[Ti]` and `sweep_df[Ti]` substitute $v = i$.

**What is asserted, per iceberg run.** On the iceberg tape, the iceberg $I$ has total $T$, peak $P$ and price $p_I$. The visible rows $u_1..u_{n_{vis}}$ sit at $p_I$, summing to $V_{vis}$. The FOK $F$ asks $R$ at a crossing limit.
- **(I0) before the cancel** (§4.2, §4.5): $I$ shows $P$ and hides $T - P$; every $u$ rests in full; the record holds one live side-channel row (`n_active` = 1).
- **(I1) the cancel** (§5.1; §4.2.1 Rule 1 counts *remaining* quantity): just before $F$, $I$ shows $0$ and hides $0$, and `n_active` = 0 (engineering, finding F4). The book is the $u$ rows in full, row for row as before. $I$'s fate is exactly one `cancellation` of $T$. The cancel prints no trade (C2).
- **(I2) the FOK** (§4.2.1 Rule 1, §4.5): the pre-check sees $M = V_{vis}$ (the cancelled iceberg contributes nothing) and $M < R$. So $F$ fills $0$ and writes no trade; the book after $F$ equals the book before $F$, row for row; $F$ never rests; its fate is a rejection or a `fok_kill` of $R$ (finding F3). At the end the book is the $u$ rows in full, $I$ is absent with no hidden quantity, and the other side is empty.
- **(I3) the no-cancel control** (§4.2.1 Rules 1–2): $M = V_{vis} + T \ge R$, so $F$ fills exactly $R$. Every fill has $F$ as aggressor and prints at $p_I$ (the only resting price); $F$ never rests. What remains on the resting side, visible plus $I$'s hidden, is $V_{vis} + T - R$. Which orders fill, and in what order, is **not** asserted: iceberg slices take new timestamps (§5.2), and that sequence is C08's and C10's to certify.

**Two models that are wrong on purpose**, run on the same tapes so section 7 can show that the dataset tells them apart from (W):
- **ignore-cancel**: the victim stays in the book. It is told apart by (B) on every draw, and by (S) whenever the victim is among the first $j+1$ touch orders.
- **swap-with-last** (finding F1): the resting rows sit in an array in entry order. The removal moves the last row into the victim's slot, and same-price rows fill in array order. It is told apart from (W) only when the last resting row is a touch order, i.e. on `decoys_first`, with $v \le \min(j, k-2)$.

The code below implements (W) as **one function of the tape rows and the cancellation set**, used unchanged for every run. It reads neither `draw.expected` nor the engine.

In [ ]:
def walk(tape, cancelled=(), model="rule"):
    """(X) + (R) + (W): one incoming order (the last real row) against the resting rows left after the
    cancellation set `cancelled`, in integer ticks.

    model="rule" is the specification. "ignore_cancel" and "swap_last" are the two wrong models of section 5,
    used only to show the dataset separates them from the rule. Returns None if the tape is not 'resting rows
    on one side, then one incoming order'."""
    sh = shape(tape)
    if sh is None:
        return None
    rest, a, rs, by = sh["rest"], sh["agg"], sh["rest_side"], sh["by_oid"]
    cancelled = [int(c) for c in cancelled]
    entry = [int(r[C_OID]) for r in rest]
    if model == "ignore_cancel":
        live = list(entry)
    elif model == "swap_last":
        live = list(entry)                    # the array, in entry order
        for c in cancelled:
            if c in live:
                pos = live.index(c)
                last = live.pop()
                if pos < len(live):
                    live[pos] = last          # the last row moves into the vacated slot
    else:
        live = [o for o in entry if o not in cancelled]
    sgn = 1 if rs == SELL else -1
    ranked = sorted(live, key=lambda o: (sgn * int(by[o][C_PRICE]), live.index(o)))
    left = {o: int(by[o][C_QTY]) for o in live}
    rho, fills = int(a[C_QTY]), []
    for o in ranked:
        if rho == 0:
            break
        if not crosses(rs, by[o][C_PRICE], a[C_PRICE]):
            break                             # ranked best first: nothing after this one crosses either
        f = min(rho, left[o])
        fills.append([o, int(a[C_OID]), int(by[o][C_PRICE]), f, True])
        left[o] -= f
        rho -= f
    book = {"BUY": [], "SELL": []}
    for o in live:
        if left[o] > 0:
            book[side_name(rs)].append([int(by[o][C_PRICE]), left[o], o])
    if rho > 0 and int(a[C_EXEC]) == NON:
        book[side_name(a[C_SIDE])].append([int(a[C_PRICE]), rho, int(a[C_OID])])
    before = {"BUY": [], "SELL": []}
    before[side_name(rs)] = sorted([int(by[o][C_PRICE]), int(by[o][C_QTY]), o] for o in live)
    survivors = [o for o in entry if o not in cancelled]
    return {"agg_oid": int(a[C_OID]), "rest_side": rs, "cancelled": cancelled,
            "victim_qty": {c: int(by[c][C_QTY]) for c in cancelled if c in by},
            "fills": fills, "filled_set": sorted({f[0] for f in fills}),
            "book": {s: sorted(v) for s, v in book.items()}, "book_before_agg": before,
            "entry_order": entry, "survivors": survivors, "touch": sh["touch"], "decoys": sh["decoys"],
            "p_touch": sh["p_touch"], "total_qty_int": int(a[C_QTY]) - rho}


def expectation(tape, cancelled):
    """(C1)-(B) of section 5, from the tape rows and the cancellation set only."""
    return walk(tape, cancelled, "rule")


def closed_form(draw, victim=None, tape=None):
    """Section 5's closed form for a ladder draw, from draw.params (decoy limits read from their tape rows)."""
    p = draw.params
    touch, j, q, pt, agg = p["resting_oids"], p["j"], p["q_each_int"], p["p_touch_int"], p["agg_oid"]
    v = p["cancel_oid"] if victim is None else victim
    S = [o for o in touch if o != v]
    by_oid = {int(r[C_OID]): r for r in real_rows(draw.tape if tape is None else tape)}
    rest = [[pt, q, o] for o in S[j:]] + [[int(by_oid[o][C_PRICE]), q, o] for o in p["decoy_oids"]]
    book = {"BUY": [], "SELL": []}
    book[side_name(p["rest_side"])] = sorted(rest)
    return {"fills": [[o, agg, pt, q, True] for o in S[:j]], "filled_set": sorted(S[:j]), "book": book}


def ice_expectation(tape, cancelled):
    """(I0)-(I3) of section 5, from the tape rows and the cancellation set only."""
    s = ice_shape(tape)
    if s is None:
        return None
    gone = s["ice_oid"] in [int(c) for c in cancelled]
    T, P, R, V = s["total"], s["peak"], s["req"], s["v_vis"]
    matchable = V + (0 if gone else T)                  # §4.2.1 Rule 1: the iceberg's REMAINING quantity
    fok_fill = R if matchable >= R else 0               # §4.5: all of R, or nothing
    vis_book = sorted([s["p_ice"], int(r[C_QTY]), int(r[C_OID])] for r in s["vis"])
    out = {"ice_oid": s["ice_oid"], "fok_oid": s["fok_oid"], "rest_side": s["rest_side"], "p_ice": s["p_ice"],
           "cancelled": gone, "n_vis": len(s["vis"]), "req": R, "matchable": matchable, "fok_fill": fok_fill,
           "pre_cancel_ice": {"visible": min(P, T), "hidden": T - min(P, T)},
           "pre_cancel_vis_book": vis_book, "n_active_around_cancel": [1, 0],
           "ice_cancel_fate": [["cancellation", T]],
           "remaining_after": V + T - fok_fill if not gone else V}
    if gone:
        book = {"BUY": [], "SELL": []}
        book[side_name(s["rest_side"])] = vis_book
        out.update({"book_before_fok": book, "final_book": book, "post_cancel_ice": {"visible": 0, "hidden": 0}})
    return out


def run_tapes(seed):
    """The (tape, cancellation plan) of every ladder run made from `seed`, keyed by run name."""
    d = DRAWN[(LADDER, seed)]
    plan = plan_of(d)
    agg = next(iter(plan))
    runs = {"draw": (d.tape, plan), "decoys_first": (DF_TAPE[seed], plan)}
    return runs, agg


def sweep_tapes(seed):
    d = DRAWN[(LADDER, seed)]
    _, agg = run_tapes(seed)
    out = {}
    for i, o in enumerate(d.params["resting_oids"], start=1):
        out[f"sweep[T{i}]"] = (d.tape, {agg: (o,)})
        out[f"sweep_df[T{i}]"] = (DF_TAPE[seed], {agg: (o,)})
    return out


def cancelled_of(plan):
    return [int(c) for cs in plan.values() for c in cs]


EXPECTED = {}
for s in range(SEED0, SEED0 + DRAWS):
    runs, _ = run_tapes(s)
    for name, (t, plan) in runs.items():
        EXPECTED[(LADDER, s, name)] = expectation(t, cancelled_of(plan))
for s in range(SEED0, SEED0 + ICE_DRAWS):
    d = DRAWN[(ICEBERG, s)]
    EXPECTED[(ICEBERG, s, "draw")] = ice_expectation(d.tape, cancelled_of(plan_of(d)))
    EXPECTED[(ICEBERG, s, "no_cancel")] = ice_expectation(d.tape, [])

for s in range(SEED0, SEED0 + 3):
    p = DRAWN[(LADDER, s)].params
    e = EXPECTED[(LADDER, s, "draw")]
    print(f"{LADDER} seed {s}: touch {p['resting_oids']}, victim {p['cancel_oid']}, j = {p['j']} -> survivors "
          f"{e['survivors']}; fills {[(f[0], f[2], f[3]) for f in e['fills']]} (oid, price_int, qty_int); rests {e['book']}")
for s in range(SEED0, SEED0 + 3):
    e, c = EXPECTED[(ICEBERG, s, "draw")], EXPECTED[(ICEBERG, s, "no_cancel")]
    print(f"{ICEBERG} seed {s}: n_vis {e['n_vis']}, pre-check sees {e['matchable']} -> FOK fills {e['fok_fill']}; "
          f"control sees {c['matchable']} -> fills {c['fok_fill']}, leaves {c['remaining_after']}")

# Section 5's closed form against the walk on the first seed's every run (the full check runs in section 7)
CF_AGREES = True
_d0 = DRAWN[(LADDER, SEED0)]
_runs0 = {**run_tapes(SEED0)[0], **sweep_tapes(SEED0)}
for name, (t, plan) in _runs0.items():
    v = cancelled_of(plan)[0]
    e = expectation(t, [v])
    cf = closed_form(_d0, victim=v, tape=t)
    ok = e["fills"] == cf["fills"] and e["book"] == cf["book"]
    CF_AGREES &= ok
    print(f"{LADDER} seed {SEED0} {name:<14}: victim {v}: walk {e['filled_set']} closed form {cf['filled_set']}  "
          f"{'agree' if ok else 'DISAGREE'}; swap-with-last model -> "
          f"{[f[0] for f in walk(t, [v], 'swap_last')['fills']]} vs rule {[f[0] for f in e['fills']]}")

## 6. Run

### Eager and jitted (wired at K3)

> **K3: this section is yours; sections 1–5 and 7 are locked (independence rule 2).** A disagreement with the expectation is escalated as a `SCOPE-CHANGE` candidate with both readings, never resolved by editing sections 5 or 7.

Section 7 consumes exactly **one name** from this section. Until it is callable, section 7 records a failing `section 6 wired` check and the notebook ends `NOT CERTIFIED`.

**`run_tape(tape, cancels_before, *, jit) -> RunRecord`**: the battery's record builder on one tape with one cancellation plan, i.e. `tests.omiecid.reference.omie_invariants.build_record(tape, cancels_before=cancels_before, jit=jit)`. It runs on one fresh book of an OPEN contract with the default configs (`OMIEProductConfig()`, `OMIEMarketConfig()`), no gate-closure expiry, and the default capacity bounds (`tape_capacity_bounds(tape)`). `cancels_before` is passed **as given**: `{order id of row i: (oid, ...)}`, and those accepted orders are cancelled (`ContractBook.cancel_order`) immediately before row $i$ is submitted. It is `plan_of(draw)` for a draw, a sweep run's own plan, or `{}` for the no-cancel control. `jit=False` runs eager (the builder wraps the run in `jax.disable_jit()`); `jit=True` runs jitted. The tape is passed **as is**: a draw's `DRAWN[(key, seed)].tape` or a `decoys_first` tape `DF_TAPE[seed]`. Nothing in the tape may be rewritten, and that includes its timestamp column.

Section 7 reads these `RunRecord` fields:
- `fills`: each `Fill`'s `passive_oid`, `aggressor_oid`, `price_int`, `qty_int`, `submission` and `row_index`;
- `accepted`: its keys, and each `AcceptedOrder.submission`;
- `rejections`: each `Rejection.oid`;
- `pre_snapshots[s]` and `post_snapshots[s - 1]` around the cancel before submission $s$: `.resting(side)` (price, qty, oid, agent, sec, nsec), `.trades`, `.hidden_qty(oid)`, `.visible_qty(oid, side)` and `.n_active`;
- `post_snapshots[s]`: `.resting(side)` and `.trades`;
- `final_snapshot`: `.resting(side)`, `.hidden_qty(oid)` and `.visible_qty(oid, side)`;
- `filled`, `cancelled_or_expired` (each `Cancellation`'s `cause` and `qty_int`), `engine_reports[s].traded` and `observed_capacity`;
- the whole record, through `run_battery` and `compare_records`.

Section 7 calls `run_tape(..., jit=True)` on every draw and every derived run it uses, and `run_tape(..., jit=False)` on the eager selection of section 3. A seed beyond the initial range (a re-draw after a discard) is prepared with `prepare(key, seed)` before the call.

In [ ]:
# ---------------------------------------------------------------------------------------
# Section 6 (K3): run_tape(tape, cancels_before, *, jit) -> RunRecord, per the contract above.
#
# The battery's record builder on one fresh book of one OPEN contract, with the default
# configs, no gate-closure expiry and the default analytic capacity bounds
# (tape_capacity_bounds(tape)). cancels_before is passed as given ({order id of row i:
# (oid, ...)}; build_record calls ContractBook.cancel_order on each oid immediately before
# row i is submitted). jit=False wraps the whole run in jax.disable_jit() (inside
# build_record); jit=True runs it jitted. The tape is passed as is: no column, including
# the timestamp column, is rewritten.
# ---------------------------------------------------------------------------------------
from omiecid_simulator.omiecid.contract import ContractStatus
from omiecid_simulator.omiecid.omie_config import OMIEMarketConfig
from tests.omiecid.reference.omie_invariants import build_record

MARKET = OMIEMarketConfig()


def run_tape(tape, cancels_before, *, jit):
    """One run of `tape` (a draw's tape or a decoys_first tape, unmodified) on a fresh OPEN book, with the
    cancellation plan `cancels_before` (plan_of(draw), a sweep run's plan, or {} for the no-cancel control)."""
    return build_record(tape, product_config=CONFIG, market_config=MARKET,
                        contract_status=ContractStatus.OPEN, cancels_before=cancels_before,
                        expire_at_gct=False, capacity_bounds=None, jit=jit)


print(f"run_tape: build_record on one OPEN contract, {type(CONFIG).__name__}() + {type(MARKET).__name__}(), "
      f"cancellation plan as given, no GCT expiry, default capacity bounds")

## 7. Certification

The oracle comparison comes first, by order id and in integer ticks. Then come the invariant battery (RB, I1–I9) as a pass/fail table, the capacity peaks and the discard rate. No check stops the notebook: every assertion is evaluated and recorded.

**What is recorded, per ladder run** (`draw`, `decoys_first`, and the sweep runs), each with its expected and observed value, so an auditor can check the exact integers from the results JSON alone:
1. `… cancel removes exactly the victim` — (C1): the observed book after the cancel, against the observed book before it minus the victim's rows (full rows: price, qty, oid, agent, sec, nsec);
2. `… victim after the cancel [visible, hidden], fate, filled` — (C1): `[[0, 0]]`, `[["cancellation", q_v]]` and 0;
3. `… trade rows written by the cancel` — (C2): 0;
4. `… book before the aggressor` — (B0): the survivors `[price_int, qty_int, oid]`;
5. `… stored timestamps of the survivors` — (T): the observed `[oid, [[sec, nsec], ...]]` in entry order, passing when each survivor holds exactly one entry and the stamps strictly increase;
6. `… filled order ids` — (S);
7. `… counterparty sequence` — (Q): `[passive oid, aggressor oid, price_int, qty_int, on the aggressor's submission]` in trade-row order;
8. `… book after` — (B): `[price_int, qty_int, oid]` per side;
9. `… battery passes` — for derived runs only. For a draw, the battery is folded into the invariant table.

**Per iceberg draw:** (I0) the iceberg's `[visible, hidden]` and the visible orders before the cancel; (I1) the removal, the iceberg's `[visible, hidden]` after it, the side-channel count `n_active` before and after (engineering, F4), the iceberg's fate, and the trade rows written by the cancel; (I2) the FOK's fills (exactly `[]`), the book and trade tape across the FOK (unchanged), the FOK's fate (rejection or `fok_kill` of $R$, F3), the engine's `traded` flag (False), and the final book with the iceberg gone. **Per no-cancel control:** (I3) the FOK's total fill ($= R$), every fill at $p_I$ with the FOK as aggressor (the **full** `fok_fills` list is recorded as the observed value, never a prefix), the FOK not resting, the remaining resting-side liquidity ($V_{vis} + T - R$), and its battery.

**Aggregate assertions:** the section-6 wiring; the dataset precondition per construction (section 4), listing every failing seed (full lists, never truncated); the closed form of section 5 against (W) on every ladder run; every derived run within §2.3's bounds and the capacity margin; **separation** — the ignore-cancel model disagrees with (W) on every ladder run, and on every sweep draw with $k \ge 3$ the swap-with-last model disagrees with (W) on at least one `sweep_df` run (F1); **coverage** — $k = 2..5$, 1–3 decoys, $j = 1$, $j = k-1$ for some $k \ge 3$, the victim first, last and interior, both resting sides; $n_{vis} = 0..3$, both resting sides, and the strong iceberg class $n_{vis} \ge 1$, $R > V_{vis} + P$ (F2); and the eager comparisons, one per required class, a missing one recorded as a failure.

The battery runs on every draw's jitted record, with `expects_trade=True` on the ladder and `expects_trade=False` on the iceberg draw: its FOK must not trade. A draw the battery marks I8 `invalid` is discarded and replaced by the next seed. For derived runs the battery is folded into one assertion each, since a derived run is not a draw. Before the full certificate table come a per-run table (all failing rows, then the first 20), the separation counts of the swap-with-last model, and the iceberg draws' observed FOK fates (printed, not asserted; F3).

In [ ]:
import traceback
from collections import defaultdict

from tests.omiecid.reference.omie_invariants import compare_records, run_battery

cert = Certification(
    cert_id=CERT_ID, notebook=NOTEBOOK, scenario="Cancellation",
    known_result="the cancelled order is gone (no visible or hidden quantity, one 'cancellation' fate of its full "
                 "size) and nothing else in the book changes; the cancellation prints no trade; the survivors keep "
                 "their stamps and queue, so the fill set is the j earliest remaining originals, filled in entry "
                 "order at the touch price; after cancelling an iceberg its hidden volume is invisible to the FOK "
                 "pre-check, so a FOK only the iceberg could have satisfied fills nothing and leaves the book "
                 "unchanged (the no-cancel control fills R exactly)",
    oracle_kind="exact-parametric", quotes=QUOTES, citations=CITATIONS,
    constructions=CONSTRUCTIONS, requested_draws=REQUESTED, runtime_forecast_s=120.0, started_at=T0,
    downstream="every scenario that cancels before matching: C11 (GCT expiry removes orders), C15 (random "
               "sessions with cancellations), C17 (cancellation fidelity); C03's priority guarantee after any "
               "cancellation; C10 (FOK pre-check reads the iceberg side-channel); Track B, where agents cancel "
               "and re-quote",
)

R_CANCEL = "§5.1 CANCELLED (by agent); §9.1 Cancel order"
R_NOTRADE = "§6.1 (match needs bid >= ask); §6.2 item 2 (opposite sides)"
R_RANK = "§6.1; §5.2 (no new timestamp on another order's cancellation)"
R_SEQ = "§6.1; §6.2 items 1, 3 (rank order, maker price)"
R_BOOK = "§4.5 (NON: book entry if unmatched); §6.2 item 1"
R_TS = "§5.2 (single global queue: unique, sequential; no re-stamp on a cancellation)"
R_ICE = "§4.2 (only the peak is shown); §4.5 ICE"
R_SLOT = "engineering invariant (card: iceberg side-channel release; OQ-20 class), finding F4"
R_FOK = "§4.2.1 Rule 1 (remaining iceberg quantity); §4.5 FOK (full or cancel, no book entry)"
R_CTRL = "§4.2.1 Rules 1-2 (FOK sees the total; multi-slice fill)"
R_DATA = "dataset precondition (section 4); §2.3"


def guarded(label, seed, fn, *args, **kw):
    """Call a section-6 function; a raise is recorded as a failed check, never propagated."""
    try:
        return fn(*args, **kw)
    except Exception as exc:  # noqa: BLE001
        cert.check(f"{label}: run raised", False, seed=seed, expected="a RunRecord", observed=type(exc).__name__,
                   detail=f"{type(exc).__name__}: {exc} | " + traceback.format_exc().splitlines()[-1], rule=R_CANCEL)
        return None


def full_rows(snap):
    """Every resting row, both sides, as sorted [price, qty, oid, agent, sec, nsec]."""
    return {side_name(sd): sorted([int(v) for v in x] for x in snap.resting(sd)) for sd in (BUY, SELL)}


def short_rows(snap):
    """Every resting row, both sides, as sorted [price, qty, oid]."""
    return {side_name(sd): sorted([int(x[0]), int(x[1]), int(x[2])] for x in snap.resting(sd)) for sd in (BUY, SELL)}


def minus_oids(rows, oids):
    return {s: [r for r in v if int(r[2]) not in set(oids)] for s, v in rows.items()}


def trade_rows_changed(a, b):
    """Number of trade-tape rows that differ between two snapshots (-1 when the arrays differ in shape)."""
    x, y = np.asarray(a.trades), np.asarray(b.trades)
    if x.shape != y.shape:
        return -1
    if x.size == 0:
        return 0
    return int(np.sum(np.any(x != y, axis=1)))


def around_cancel(rec):
    """(submission of the last row, snapshot just before the cancel, snapshot just after it), or None."""
    s = int(rec.n_submissions) - 1
    if s < 1 or s >= len(rec.pre_snapshots) or s - 1 >= len(rec.post_snapshots):
        return None
    return s, rec.post_snapshots[s - 1], rec.pre_snapshots[s]


def observe_ladder(rec, e):
    """What the engine did on a ladder run, read from the record, in the expectation's terms."""
    ac = around_cancel(rec)
    if ac is None:
        return None
    s, bc, cc = ac
    agg, rs, victims = e["agg_oid"], e["rest_side"], e["cancelled"]
    a_sub = int(rec.accepted[agg].submission) if agg in rec.accepted else None
    fills = sorted(rec.fills, key=lambda f: (int(f.submission), int(f.row_index)))
    seq = [[int(f.passive_oid), int(f.aggressor_oid), int(f.price_int), int(f.qty_int),
            a_sub is not None and int(f.submission) == a_sub] for f in fills]
    stamps = defaultdict(list)
    for x in cc.resting(rs):
        stamps[int(x[2])].append([int(x[4]), int(x[5])])
    return {
        "removal_obs": full_rows(cc), "removal_exp": minus_oids(full_rows(bc), victims),
        "victim": {"after_cancel [visible, hidden]": [[cc.visible_qty(v, rs), cc.hidden_qty(v)] for v in victims],
                   "fate [cause, qty_int]": [[[c.cause, int(c.qty_int)] for c in rec.cancelled_or_expired.get(v, ())]
                                             for v in victims],
                   "filled": [int(rec.filled.get(v, 0)) for v in victims]},
        "trades_from_cancel": trade_rows_changed(bc, cc),
        "book_before_agg": short_rows(cc),
        "ts": [[o, stamps.get(o, [])] for o in e["survivors"]],
        "seq": seq, "filled_set": sorted({x[0] for x in seq}), "book": short_rows(rec.final_snapshot),
        "agg_submission_is_last": a_sub == s,
    }


def ts_ok(ts):
    """(T): one stored entry per survivor, stamps strictly increasing in entry order."""
    if ts is None or any(len(st) != 1 for _, st in ts):
        return False
    stamps = [tuple(st[0]) for _, st in ts]
    return all(a < b for a, b in zip(stamps[:-1], stamps[1:]))


def check_ladder(label, seed, rec, e):
    """(C1), (C2), (B0), (T), (S), (Q), (B) of section 5. Returns (observation, all passed)."""
    o = observe_ladder(rec, e)
    if o is None:
        cert.check(f"{label}: record has the cancel window (a submission after the resting rows)", False,
                   seed=seed, expected=">= 2 submissions", observed=int(rec.n_submissions), unit="count",
                   rule=R_CANCEL)
        return None, False
    ok = cert.check_equal(f"{label}: cancel removes exactly the victim (book after the cancel == book before it "
                          f"minus the victim; full rows [price, qty, oid, agent, sec, nsec])",
                          o["removal_obs"], o["removal_exp"], seed=seed, unit="other", rule=R_CANCEL)
    want_v = {"after_cancel [visible, hidden]": [[0, 0] for _ in e["cancelled"]],
              "fate [cause, qty_int]": [[["cancellation", e["victim_qty"][v]]] for v in e["cancelled"]],
              "filled": [0 for _ in e["cancelled"]]}
    ok &= cert.check_equal(f"{label}: victim {e['cancelled']} after the cancel [visible, hidden], fate, filled",
                           o["victim"], want_v, seed=seed, unit="other", rule=R_CANCEL)
    ok &= cert.check_equal(f"{label}: trade rows written by the cancel", o["trades_from_cancel"], 0, seed=seed,
                           unit="count", rule=R_NOTRADE)
    ok &= cert.check_equal(f"{label}: book before the aggressor [price_int, qty_int, oid]", o["book_before_agg"],
                           e["book_before_agg"], seed=seed, unit="other", rule=R_CANCEL)
    ok &= cert.check(f"{label}: stored timestamps of the survivors unique and increasing in entry order",
                     ts_ok(o["ts"]), seed=seed,
                     expected={"survivors_in_entry_order": e["survivors"],
                               "stamps": "one [sec, nsec] each, strictly increasing"},
                     observed=o["ts"], unit="other", rule=R_TS)
    ok &= cert.check_equal(f"{label}: filled order ids", o["filled_set"], e["filled_set"], seed=seed, unit="other",
                           rule=R_RANK)
    ok &= cert.check_equal(f"{label}: counterparty sequence [passive, aggressor, price_int, qty_int, on aggressor's "
                           f"submission]", o["seq"], e["fills"], seed=seed, unit="other", rule=R_SEQ)
    ok &= cert.check_equal(f"{label}: book after [price_int, qty_int, oid]", o["book"], e["book"], seed=seed,
                           unit="other", rule=R_BOOK)
    return o, bool(ok)


def observe_iceberg(rec, e):
    ac = around_cancel(rec)
    if ac is None:
        return None
    s, bc, cc = ac
    ice, fok, rs = e["ice_oid"], e["fok_oid"], e["rest_side"]
    pf, fin = rec.post_snapshots[s], rec.final_snapshot
    f_fills = [[int(f.passive_oid), int(f.aggressor_oid), int(f.price_int), int(f.qty_int)]
               for f in sorted(rec.fills, key=lambda f: (int(f.submission), int(f.row_index))) if int(f.submission) == s]
    rejected = any(int(r.oid) == fok for r in rec.rejections)
    fate = "rejected" if rejected else [[c.cause, int(c.qty_int)] for c in rec.cancelled_or_expired.get(fok, ())]
    other = SELL if rs == BUY else BUY
    return {
        "pre_cancel_ice": {"visible": bc.visible_qty(ice, rs), "hidden": bc.hidden_qty(ice)},
        "pre_cancel_vis_book": sorted([int(x[0]), int(x[1]), int(x[2])] for x in bc.resting(rs) if int(x[2]) != ice),
        "n_active": [int(bc.n_active), int(cc.n_active)],
        "removal_obs": full_rows(cc), "removal_exp": minus_oids(full_rows(bc), [ice]),
        "post_cancel_ice": {"visible": cc.visible_qty(ice, rs), "hidden": cc.hidden_qty(ice)},
        "ice_fate": [[c.cause, int(c.qty_int)] for c in rec.cancelled_or_expired.get(ice, ())],
        "trades_from_cancel": trade_rows_changed(bc, cc),
        "fok_fills": f_fills,
        "fok_window": {"book": full_rows(pf), "trade_rows_changed": trade_rows_changed(cc, pf)},
        "fok_window_before": {"book": full_rows(cc), "trade_rows_changed": 0},
        "fok_fate": fate,
        "fok_traded": bool(rec.engine_reports[s].traded) if s < len(rec.engine_reports) else None,
        "final": {"book": short_rows(fin), "iceberg [visible, hidden]": [fin.visible_qty(ice, rs), fin.hidden_qty(ice)],
                  "fok resting": fin.visible_qty(fok, other)},
        "remaining": int(sum(int(x[1]) for x in fin.resting(rs)) + fin.hidden_qty(ice)),
    }


def check_iceberg(label, seed, rec, e):
    """(I0)-(I2) of section 5 on an iceberg draw. Returns (observation, all passed)."""
    o = observe_iceberg(rec, e)
    if o is None:
        cert.check(f"{label}: record has the cancel window", False, seed=seed, expected=">= 2 submissions",
                   observed=int(rec.n_submissions), unit="count", rule=R_CANCEL)
        return None, False
    ok = cert.check_equal(f"{label}: before the cancel, the iceberg shows P and hides T - P", o["pre_cancel_ice"],
                          e["pre_cancel_ice"], seed=seed, unit="qty_int", rule=R_ICE)
    ok &= cert.check_equal(f"{label}: before the cancel, the visible orders rest in full [price_int, qty_int, oid]",
                           o["pre_cancel_vis_book"], e["pre_cancel_vis_book"], seed=seed, unit="other", rule=R_BOOK)
    ok &= cert.check_equal(f"{label}: cancel removes exactly the iceberg (full rows)", o["removal_obs"],
                           o["removal_exp"], seed=seed, unit="other", rule=R_CANCEL)
    ok &= cert.check_equal(f"{label}: after the cancel, the iceberg shows and hides nothing", o["post_cancel_ice"],
                           e["post_cancel_ice"], seed=seed, unit="qty_int", rule=R_CANCEL + "; " + R_FOK)
    ok &= cert.check_equal(f"{label}: side-channel slot released: live count n_active just before / just after "
                           f"the cancel (engineering, F4)", o["n_active"], e["n_active_around_cancel"], seed=seed,
                           unit="count", rule=R_SLOT)
    ok &= cert.check_equal(f"{label}: iceberg fate [cause, qty_int]", o["ice_fate"], e["ice_cancel_fate"], seed=seed,
                           unit="other", rule=R_CANCEL)
    ok &= cert.check_equal(f"{label}: trade rows written by the cancel", o["trades_from_cancel"], 0, seed=seed,
                           unit="count", rule=R_NOTRADE)
    ok &= cert.check_equal(f"{label}: FOK fills [passive, aggressor, price_int, qty_int]", o["fok_fills"], [],
                           seed=seed, unit="other", rule=R_FOK)
    ok &= cert.check_equal(f"{label}: the FOK leaves the book (full rows) and the trade tape unchanged",
                           o["fok_window"], o["fok_window_before"], seed=seed, unit="other", rule=R_FOK)
    fate_ok = o["fok_fate"] == "rejected" or o["fok_fate"] == [["fok_kill", e["req"]]]
    ok &= cert.check(f"{label}: FOK fate is a rejection or one fok_kill of R (F3)", fate_ok, seed=seed,
                     expected=["rejected", [["fok_kill", e["req"]]]], observed=o["fok_fate"], unit="other",
                     rule=R_FOK)
    ok &= cert.check_equal(f"{label}: engine report: the FOK traded", o["fok_traded"], False, seed=seed, unit="bool",
                           rule=R_FOK)
    ok &= cert.check_equal(f"{label}: final book [price_int, qty_int, oid], iceberg [visible, hidden], FOK resting",
                           o["final"], {"book": e["final_book"], "iceberg [visible, hidden]": [0, 0], "fok resting": 0},
                           seed=seed, unit="other", rule=R_CANCEL + "; " + R_FOK)
    return o, bool(ok)


def check_control(label, seed, rec, e):
    """(I3) of section 5 on the no-cancel control."""
    o = observe_iceberg(rec, e)
    if o is None:
        cert.check(f"{label}: record has the FOK's submission", False, seed=seed, expected=">= 2 submissions",
                   observed=int(rec.n_submissions), unit="count", rule=R_CTRL)
        return None, False
    got = int(sum(f[3] for f in o["fok_fills"] if f[1] == e["fok_oid"]))
    ok = cert.check_equal(f"{label}: FOK fills exactly R (qty_int)", got, e["fok_fill"], seed=seed, unit="qty_int",
                          rule=R_CTRL)
    bad = [f for f in o["fok_fills"] if f[1] != e["fok_oid"] or f[2] != e["p_ice"]]
    ok &= cert.check(f"{label}: every FOK fill has the FOK as aggressor and prints at the iceberg's price",
                     not bad and bool(o["fok_fills"]), seed=seed,
                     expected={"aggressor": e["fok_oid"], "price_int": e["p_ice"]}, observed={"departing": bad, "fok_fills": o["fok_fills"]},
                     unit="other", rule=R_CTRL + "; §6.2 item 3")
    ok &= cert.check_equal(f"{label}: FOK resting at the end (qty_int)", o["final"]["fok resting"], 0, seed=seed,
                           unit="qty_int", rule=R_FOK)
    ok &= cert.check_equal(f"{label}: resting-side liquidity left (visible + iceberg hidden, qty_int)", o["remaining"],
                           e["remaining_after"], seed=seed, unit="qty_int", rule=R_CTRL)
    return o, bool(ok)


def battery_check(label, seed, rec, expects_trade, rule):
    rep = run_battery(rec, expects_trade=expects_trade)
    return cert.check(f"{label}: battery (RB, I1-I9) passes", rep.passed and rep.valid, seed=seed,
                      expected="pass / not_applicable", observed=rep.statuses(), unit="other", rule=rule)


def eager_compare(label, seed, tape, plan, rec):
    """Run `tape` eager with `plan` and compare with its jitted record; recorded as one eager_jit assertion."""
    rec_e = guarded(f"{label} (eager)", seed, run_tape, tape, plan, jit=False)
    if rec is None or rec_e is None:
        diff = ["jitted run missing" if rec is None else "eager run missing"]
    else:
        diff = list(compare_records(rec_e, rec))
    cert.set_eager_jit(seed, diff == [], detail=f"{label}: " + (", ".join(diff) if diff else "identical"))
    return diff == []


def margin_breaches(tape, bounds):
    """The capacity-validity precondition, checked from the tape before it is run."""
    lim = {"resting_per_side": ENVELOPE.max_resting_per_side, "trades": ENVELOPE.max_trades,
           "hops": ENVELOPE.max_hops, "concurrent_icebergs": ENVELOPE.max_concurrent_icebergs}
    bad = [f"{k} bound {bounds.get(k)} > {m}" for k, m in lim.items() if int(bounds.get(k, 0)) > m]
    if max_slices(tape) > SLICE_MARGIN:
        bad.append(f"ceil(qty/peak) {max_slices(tape)} > {SLICE_MARGIN}")
    return bad


# ---- section 5's own closed form agrees with the walk on the first seed ------------------------------
cert.check("section 5: closed form agrees with the walk (W) on every run of the first ladder seed", CF_AGREES,
           expected=True, observed=CF_AGREES, unit="bool", rule=R_RANK)

wired = callable(run_tape)
cert.check("section 6 wired (run_tape defined)", wired, expected=True, observed=wired, unit="bool",
           detail="" if wired else "section 6 is still the K2 stub", rule=R_CANCEL)

PER_RUN, DATA_BAD, CLOSED_BAD, SEP_BAD = [], defaultdict(dict), [], defaultdict(list)
DERIVED_BAD, CLASS_COUNT, COVER = [], Counter(), defaultdict(Counter)
EAGER_DONE, FOK_FATES = {}, Counter()
SWAP_SEP = Counter()           # runs where the swap-with-last model disagrees with (W), per run kind
SWAP_SWEEP_BAD = []            # sweep draws with k >= 3 where no sweep_df run separates the swap model
PEAK = {"hop_bound": 0, "max_slices": 0}


def eager_once(tag, label, seed, tape, plan, rec):
    """Force one eager comparison per tag (class or variant); later calls with the same tag do nothing."""
    if tag not in EAGER_DONE:
        EAGER_DONE[tag] = seed
        eager_compare(label, seed, tape, plan, rec)


def separates(t, plan, model):
    v = cancelled_of(plan)
    a, b = walk(t, v, "rule"), walk(t, v, model)
    return a["fills"] != b["fills"] or a["book"] != b["book"]


def run_ladder_variant(seed, name, t, plan, lbl, draw):
    """One jitted ladder run (draw or derived), its checks, and its closed-form / separation bookkeeping."""
    e = expectation(t, cancelled_of(plan))
    if name != "draw":
        bad = margin_breaches(t, synth.tape_capacity_bounds(t))
        if bad or not bounds_ok(t) or e is None:
            DERIVED_BAD.append(f"{lbl}: " + ("; ".join(bad) if bad else "outside §2.3 / not a ladder tape"))
            return None, e
    cf = closed_form(draw, victim=cancelled_of(plan)[0], tape=t)
    if e is None or e["fills"] != cf["fills"] or e["book"] != cf["book"]:
        CLOSED_BAD.append(f"seed {seed} {name}")
    if not separates(t, plan, "ignore_cancel"):
        SEP_BAD["ignore_cancel"].append(f"seed {seed} {name}")
    kind = name.split("[")[0]
    SWAP_SEP[kind, "runs"] += 1
    SWAP_SEP[kind, "separated"] += int(separates(t, plan, "swap_last"))
    rec = guarded(f"{lbl} (jit)", seed, run_tape, t, plan, jit=True)
    return rec, e


def run_ladder():
    valid, seed, n = 0, SEED0, N_DRAWS[LADDER]
    while valid < n and seed < SEED0 + 2 * n:
        draw = DRAWN.get((LADDER, seed)) or prepare(LADDER, seed)
        margin = margin_breaches(draw.tape, dict(draw.capacity_bounds))
        if margin:
            cert.discard(seed, f"{LADDER}: pre-run capacity margin: " + "; ".join(margin))
            seed += 1
            continue
        bad = precondition_ladder(draw)
        if shape(draw.tape) is None or seed not in DF_TAPE:
            cert.check(f"{LADDER} seed {seed}: expectation defined", False, seed=seed, expected="a ladder tape",
                       observed=bad, unit="other", rule=R_DATA)
            DATA_BAD[LADDER][seed] = bad
            seed += 1
            continue
        runs, _ = run_tapes(seed)
        t, plan = runs["draw"]
        lbl = f"{LADDER} seed {seed}"
        rec, e = run_ladder_variant(seed, "draw", t, plan, lbl, draw)
        if rec is not None and not cert.add_battery(run_battery(rec, expects_trade=True), seed=seed):
            seed += 1   # I8 invalid: discarded, re-drawn with the next seed
            continue
        valid += 1
        cert.record_case(seed, draw.tape, draw.params)
        PEAK["hop_bound"] = max(PEAK["hop_bound"], int(draw.hop_bound))
        PEAK["max_slices"] = max(PEAK["max_slices"], max_slices(draw.tape))
        if bad:
            DATA_BAD[LADDER][seed] = bad
        p = draw.params
        k, j = p.get("k"), p.get("j")
        vpos = p["resting_oids"].index(p["cancel_oid"]) + 1 if p.get("cancel_oid") in p.get("resting_oids", []) else -1
        COVER[LADDER]["k", k] += 1
        COVER[LADDER]["decoys", p.get("n_decoys")] += 1
        COVER[LADDER]["j=1"] += int(j == 1)
        COVER[LADDER]["j=k-1, k>=3"] += int((k or 0) >= 3 and j == (k or 0) - 1)
        COVER[LADDER]["victim first"] += int(vpos == 1)
        COVER[LADDER]["victim last"] += int(vpos == k)
        COVER[LADDER]["victim interior (k>=3)"] += int(1 < vpos < (k or 0))
        side = side_name(p.get("rest_side", -1))
        CLASS_COUNT[(LADDER, "draw", side)] += 1
        if rec is not None:
            cert.record_capacity({**rec.observed_capacity, "hops": None})   # hops: unobserved by the engine
            o, ok = check_ladder(lbl, seed, rec, e)
            PER_RUN.append({"seed": seed, "run": "draw", "e": e, "o": o, "ok": ok})
            if valid <= EAGER_DRAWS:
                eager_compare(lbl, seed, t, plan, rec)
                EAGER_DONE.setdefault((LADDER, "draw", side), seed)
            else:
                eager_once((LADDER, "draw", side), lbl, seed, t, plan, rec)

        # ---- decoys_first: every valid draw ---------------------------------------------------------
        t, plan = runs["decoys_first"]
        l2 = f"{lbl} decoys_first"
        rd, ed = run_ladder_variant(seed, "decoys_first", t, plan, l2, draw)
        if rd is not None:
            battery_check(l2, seed, rd, True, R_RANK)
            o, ok = check_ladder(l2, seed, rd, ed)
            PER_RUN.append({"seed": seed, "run": "decoys_first", "e": ed, "o": o, "ok": ok})
            eager_once((LADDER, "decoys_first", side), l2, seed, t, plan, rd)

        # ---- the victim sweep: the first SWEEP_DRAWS valid draws -------------------------------------
        if valid <= SWEEP_DRAWS:
            sep_df = False
            for name, (t, plan) in sweep_tapes(seed).items():
                l3 = f"{lbl} {name}"
                rs_, es = run_ladder_variant(seed, name, t, plan, l3, draw)
                if name.startswith("sweep_df") and separates(t, plan, "swap_last"):
                    sep_df = True
                if rs_ is None:
                    continue
                battery_check(l3, seed, rs_, True, R_RANK)
                o, ok = check_ladder(l3, seed, rs_, es)
                PER_RUN.append({"seed": seed, "run": name, "e": es, "o": o, "ok": ok})
                eager_once((LADDER, name.split("[")[0]), l3, seed, t, plan, rs_)
            COVER[LADDER]["sweep draws with k>=3"] += int((k or 0) >= 3)
            if (k or 0) >= 3 and not sep_df:
                SWAP_SWEEP_BAD.append(seed)
        seed += 1
    return valid


def run_iceberg():
    valid, seed, n = 0, SEED0, N_DRAWS[ICEBERG]
    while valid < n and seed < SEED0 + 2 * n:
        draw = DRAWN.get((ICEBERG, seed)) or prepare(ICEBERG, seed)
        margin = margin_breaches(draw.tape, dict(draw.capacity_bounds))
        if margin:
            cert.discard(seed, f"{ICEBERG}: pre-run capacity margin: " + "; ".join(margin))
            seed += 1
            continue
        bad = precondition_iceberg(draw)
        plan = plan_of(draw)
        e = EXPECTED.get((ICEBERG, seed, "draw")) or ice_expectation(draw.tape, cancelled_of(plan))
        ec = EXPECTED.get((ICEBERG, seed, "no_cancel")) or ice_expectation(draw.tape, [])
        if e is None or ec is None:
            cert.check(f"{ICEBERG} seed {seed}: expectation defined", False, seed=seed, expected="an iceberg tape",
                       observed=bad, unit="other", rule=R_DATA)
            DATA_BAD[ICEBERG][seed] = bad
            seed += 1
            continue
        lbl = f"{ICEBERG} seed {seed}"
        rec = guarded(f"{lbl} (jit)", seed, run_tape, draw.tape, plan, jit=True)
        if rec is not None and not cert.add_battery(run_battery(rec, expects_trade=False), seed=seed):
            seed += 1
            continue
        valid += 1
        cert.record_case(seed, draw.tape, draw.params)
        PEAK["hop_bound"] = max(PEAK["hop_bound"], int(draw.hop_bound))
        PEAK["max_slices"] = max(PEAK["max_slices"], max_slices(draw.tape))
        if bad:
            DATA_BAD[ICEBERG][seed] = bad
        s_ = ice_shape(draw.tape)
        side = side_name(s_["rest_side"])
        nv = "n_vis=0" if s_["vis"].shape[0] == 0 else "n_vis>=1"
        COVER[ICEBERG]["n_vis", len(s_["vis"])] += 1
        COVER[ICEBERG]["strong: n_vis>=1 and R > V_vis + P"] += int(len(s_["vis"]) >= 1
                                                                   and s_["req"] > s_["v_vis"] + s_["peak"])
        CLASS_COUNT[(ICEBERG, "draw", side, nv)] += 1
        if rec is not None:
            cert.record_capacity({**rec.observed_capacity, "hops": None})
            o, ok = check_iceberg(lbl, seed, rec, e)
            if o is not None:
                FOK_FATES[str(o["fok_fate"]) if o["fok_fate"] == "rejected" else
                          "fok_kill" if o["fok_fate"] and o["fok_fate"][0][0] == "fok_kill" else str(o["fok_fate"])] += 1
            PER_RUN.append({"seed": seed, "run": "iceberg", "e": e, "o": o, "ok": ok})
            if valid <= EAGER_DRAWS:
                eager_compare(lbl, seed, draw.tape, plan, rec)
                EAGER_DONE.setdefault((ICEBERG, "draw", side, nv), seed)
            else:
                eager_once((ICEBERG, "draw", side, nv), lbl, seed, draw.tape, plan, rec)

        # ---- the no-cancel control: every valid draw -------------------------------------------------
        lc = f"{lbl} no_cancel"
        rc = guarded(f"{lc} (jit)", seed, run_tape, draw.tape, {}, jit=True)
        if rc is not None:
            battery_check(lc, seed, rc, True, R_CTRL)
            o, ok = check_control(lc, seed, rc, ec)
            PER_RUN.append({"seed": seed, "run": "no_cancel", "e": ec, "o": o, "ok": ok})
            eager_once((ICEBERG, "no_cancel", side), lc, seed, draw.tape, {}, rc)
        seed += 1
    return valid


VALID = {}
if wired:
    VALID[LADDER] = run_ladder()
    VALID[ICEBERG] = run_iceberg()

# ---- aggregate assertions ------------------------------------------------------------------------
for key in CONSTRUCTIONS:
    bad = DATA_BAD.get(key, {})
    cert.check(f"dataset: every valid {key} draw is as section 4 describes (precondition)", wired and not bad,
               expected=[], observed={str(k): v for k, v in bad.items()}, unit="other",
               detail=(f"{len(bad)} draws depart; first: {list(bad)[:10]}" if bad
                       else f"{VALID.get(key, 0)}/{VALID.get(key, 0)} draws"), rule=R_DATA)
cert.check("section 5: closed form agrees with the walk (W) on every ladder run", wired and not CLOSED_BAD,
           expected=[], observed=CLOSED_BAD, unit="other", rule=R_RANK)
cert.check("dataset: every derived run is within §2.3's bounds and the capacity margin", wired and not DERIVED_BAD,
           expected=[], observed=DERIVED_BAD, unit="other", rule="§2.3; capacity-validity precondition")
cert.check("dataset separates the rule from the ignore-cancel model on every ladder run",
           wired and not SEP_BAD["ignore_cancel"], expected=[], observed=SEP_BAD["ignore_cancel"], unit="other",
           rule=R_CANCEL)
cert.check("dataset separates the rule from the swap-with-last model on every sweep draw with k >= 3 (F1)",
           wired and not SWAP_SWEEP_BAD and COVER[LADDER]["sweep draws with k>=3"] > 0, expected=[],
           observed={"draws without a separating sweep_df run": SWAP_SWEEP_BAD,
                     "sweep draws with k>=3": COVER[LADDER]["sweep draws with k>=3"]},
           unit="other", rule=R_RANK)

_want = {f"k={k}": ">= 1" for k in K_RANGE}
_want.update({f"decoys={d}": ">= 1" for d in DECOY_RANGE})
_want.update({c: ">= 1" for c in ("j=1", "j=k-1, k>=3", "victim first", "victim last", "victim interior (k>=3)",
                                  "resting BUY", "resting SELL")})
_c = COVER[LADDER]
_got = {f"k={k}": _c["k", k] for k in K_RANGE}
_got.update({f"decoys={d}": _c["decoys", d] for d in DECOY_RANGE})
_got.update({c: _c[c] for c in ("j=1", "j=k-1, k>=3", "victim first", "victim last", "victim interior (k>=3)")})
_got.update({"resting BUY": CLASS_COUNT[(LADDER, "draw", "BUY")], "resting SELL": CLASS_COUNT[(LADDER, "draw", "SELL")]})
_missing = [k for k, v in _got.items() if v == 0]
cert.check(f"dataset: {LADDER} covers k = 2..5, 1..3 decoys, j = 1, j = k - 1, the victim first / last / interior "
           f"and both resting sides", wired and not _missing, expected=_want, observed=_got, unit="count",
           detail=f"missing: {_missing}" if _missing else "", rule=R_DATA)
_ci = COVER[ICEBERG]
_want_i = {f"n_vis={v}": ">= 1" for v in NVIS_RANGE}
_want_i.update({"strong: n_vis>=1 and R > V_vis + P": ">= 1", "resting BUY": ">= 1", "resting SELL": ">= 1"})
_got_i = {f"n_vis={v}": _ci["n_vis", v] for v in NVIS_RANGE}
_got_i["strong: n_vis>=1 and R > V_vis + P"] = _ci["strong: n_vis>=1 and R > V_vis + P"]
for sd in ("BUY", "SELL"):
    _got_i[f"resting {sd}"] = sum(CLASS_COUNT[(ICEBERG, "draw", sd, nv)] for nv in ("n_vis=0", "n_vis>=1"))
_missing_i = [k for k, v in _got_i.items() if v == 0]
cert.check(f"dataset: {ICEBERG} covers n_vis = 0..3, both resting sides and the strong class (F2)",
           wired and not _missing_i, expected=_want_i, observed=_got_i, unit="count",
           detail=f"missing: {_missing_i}" if _missing_i else "", rule=R_DATA)

# ---- eager: every required class and variant must have had one comparison -------------------------
REQUIRED_EAGER = ([(LADDER, v, sd) for v in ("draw", "decoys_first") for sd in ("BUY", "SELL")]
                  + [(LADDER, "sweep"), (LADDER, "sweep_df")]
                  + [(ICEBERG, "draw", sd, nv) for sd in ("BUY", "SELL") for nv in ("n_vis=0", "n_vis>=1")]
                  + [(ICEBERG, "no_cancel", sd) for sd in ("BUY", "SELL")])
for tag in REQUIRED_EAGER:
    if tag not in EAGER_DONE:
        cert.set_eager_jit(None, False, detail=f"MISSING: {'/'.join(tag)} was never compared eager")


# ---- the per-run table: failing rows first, then the first 20 ---------------------------------------
def _fmt(r):
    e, o = r["e"], r["o"]
    if r["run"] in ("iceberg", "no_cancel"):
        obs = "—" if o is None else (o["fok_fills"] if r["run"] == "iceberg" else
                                     sum(f[3] for f in o["fok_fills"]))
        exp = [] if r["run"] == "iceberg" else e["fok_fill"]
        return (f"| {r['seed']} | {r['run']} | iceberg {e['ice_oid']} | FOK fill {exp} | {obs} | — | — | "
                f"{'—' if o is None else o['n_active']} | {'PASS' if r['ok'] else 'FAIL'} |")
    return (f"| {r['seed']} | {r['run']} | {e['cancelled']} | {e['filled_set']} | "
            f"{'—' if o is None else o['filled_set']} | {[f[0] for f in e['fills']]} | "
            f"{'—' if o is None else [f[0] for f in o['seq']]} | "
            f"{'—' if o is None else ('yes' if ts_ok(o['ts']) else 'NO')} | {'PASS' if r['ok'] else 'FAIL'} |")


_hdr = ["| seed | run | cancelled | filled exp | filled obs | sequence exp | sequence obs | ts ok / n_active | row |",
        "|" + "---|" * 9]
_fail = [r for r in PER_RUN if not r["ok"]]
display(Markdown(f"**Per-run order ids** ({len(PER_RUN)} runs; {len(_fail)} failing)\n\n"
                 + "\n".join(_hdr + [_fmt(r) for r in _fail[:50]] + [_fmt(r) for r in PER_RUN[:20] if r["ok"]])))
print("swap-with-last model vs (W) (F1; printed, the sweep_df assertion above is the asserted part): "
      + ", ".join(f"{k}: {SWAP_SEP[k, 'separated']}/{SWAP_SEP[k, 'runs']} runs separated"
                  for k in ("draw", "decoys_first", "sweep", "sweep_df")))
print(f"iceberg draws: observed FOK fates (F3; printed, not asserted beyond 'rejected or fok_kill of R'): {dict(FOK_FATES)}")
print(f"valid draws: {VALID}; classes: {dict(CLASS_COUNT)}")
print(f"eager comparisons forced: {len(EAGER_DONE)} tags; "
      f"{sum(1 for a in cert.payload()['assertions'] if a['kind'] == 'eager_jit')} eager_jit assertions in all")
print(f"pre-run margins: largest hop bound {PEAK['hop_bound']} (<= {ENVELOPE.max_hops}), "
      f"largest ceil(qty/peak) {PEAK['max_slices']} (<= {SLICE_MARGIN})")

display(Markdown(cert.table()))

## 8. Verdict

Emit the results JSON (validated against `results.schema.json`) and print exactly one word:
`PASS` or `NOT CERTIFIED`. On `NOT CERTIFIED` the smallest failing case is frozen as a fixture.

In [ ]:
RESULTS_PATH = cert.emit()
print(cert.verdict_line())
print(f"results -> {RESULTS_PATH}")

In [ ]:
# Section 9 ("Why this certification failed") is rendered here only on NOT CERTIFIED; it is
# persisted as the last markdown cell at K6-F by cert_support.append_failure_section.
_section9 = cert.failure_section_markdown()
if _section9:
    display(Markdown(_section9))